# Staggered DID, Matching Methods, and Monte Carlo Simulation

本 notebook 使用 Stata 代码完成交错处理 DID、匹配方法和 Monte Carlo 模拟的大作业。

本作业的核心目标是：在一个自己设计的交错处理 DGP 下，比较传统 TWFE、匹配类方法和现代 DID 方法在不同数据环境中的表现。

本文使用的主要设定如下：

| 项目 | 设定 |
|---|---|
| 个体数 | $N = 500$ |
| 时间期数 | $T = 10$ |
| Monte Carlo 重复次数 | $R = 1000$ |
| 处理批次 | $G_i = 4, 6, 8$ |
| 对照组 | $G_i = 0$，即 never-treated |
| DGP 场景 | A、B、C、D 四个场景 |

本文比较的估计方法包括：

| 类别 | 方法 |
|---|---|
| 传统 DID | TWFE 双向固定效应 |
| 匹配 / 合成类方法 | 同期 PSM-DID |
| 匹配 / 合成类方法 | 滚动 PSM-DID |
| 匹配 / 合成类方法 | 合成控制法 SCM |
| 现代 DID | Callaway and Sant’Anna DID |
| 现代 DID | Sun and Abraham 事件研究估计 |

本文不使用合成双重差分 SDID，也不使用 Borusyak-Jaravel-Spiess 的 did_imputation 方法。

## Part 0. Notebook Setup

本部分用于完成 notebook 的基础设置，包括：

1. 清空 Stata 当前环境；
2. 设置随机种子；
3. 设置全局参数；
4. 安装后续估计方法所需的 Stata 命令。

为了保证整份 notebook 的参数统一，本文将样本量、时间期数和 Monte Carlo 重复次数设置为全局变量：

```stata
global N 500
global T 10
global R 1000

In [1]:
****************************************************
* Part 0.1：基础环境设置
****************************************************

clear all
set more off
set seed 20260529

global N 500
global T 10
global R 1000

* 显示当前基础设定，方便检查
display "=============================================="
display "基础参数设置"
display "=============================================="
display "个体数量 N = $N"
display "时间期数 T = $T"
display "Monte Carlo 重复次数 R = $R"
display "总观测数 N*T = " $N * $T
display "=============================================="









基础参数设置


个体数量 N = 500

时间期数 T = 10

Monte Carlo 重复次数 R = 1000

总观测数 N*T = 5000



In [2]:
****************************************************
* Part 0.2：安装所需 Stata 命令
* 目的：安装后续估计方法需要用到的外部命令
****************************************************

* 合成控制法
ssc install synth, replace

* Callaway and Sant'Anna DID 及其依赖
ssc install drdid, replace
ssc install csdid, replace

* Sun and Abraham 事件研究估计及其依赖
ssc install require, replace
ssc install ftools, replace
ssc install reghdfe, replace
ssc install avar, replace
ssc install eventstudyinteract, replace


checking synth consistency and verifying not already installed...
all files already exist and are up to date.

checking drdid consistency and verifying not already installed...
all files already exist and are up to date.

checking csdid consistency and verifying not already installed...
all files already exist and are up to date.

checking require consistency and verifying not already installed...
all files already exist and are up to date.

checking ftools consistency and verifying not already installed...
all files already exist and are up to date.

checking reghdfe consistency and verifying not already installed...
all files already exist and are up to date.

checking avar consistency and verifying not already installed...
all files already exist and are up to date.

checking eventstudyinteract consistency and verifying not already installed...
all files already exist and are up to date.


In [3]:
****************************************************
* Part 0.3：检查命令是否安装成功
****************************************************

which synth
which drdid
which csdid
which require
which ftools
which reghdfe
which avar
which eventstudyinteract


C:\Users\smoothyzhang\ado\plus\s\synth.ado
*! version 0.0.8  Jens Hainmueller 04/29/2026

C:\Users\smoothyzhang\ado\plus\d\drdid.ado

C:\Users\smoothyzhang\ado\plus\c\csdid.ado
*! v1.81  by pedro Sant'Anna. Compatibility checks
*! v1.8  by FRA. Trim

C:\Users\smoothyzhang\ado\plus\r\require.ado
*! version 1.3.1 19sep2023

C:\Users\smoothyzhang\ado\plus\f\ftools.ado
*! version 2.50.0 09jan2026

C:\Users\smoothyzhang\ado\plus\r\reghdfe.ado
*! version 6.13.1 10Jan2026

C:\Users\smoothyzhang\ado\plus\a\avar.ado
*! avar 1.0.07  28July2015
*! authors cfb/mes
*! shell based on ranktest by fk/mes
*! see end of file for version comments

C:\Users\smoothyzhang\ado\plus\e\eventstudyinteract.ado
*! version 0.1  24jan2022  Liyang Sun, lsun20@mit.edu


## Part 1. DGP Design

本部分设计交错处理 DID 的数据生成过程，也就是 Data Generating Process，简称 DGP。

我们构造一个平衡面板数据集：

$$
i = 1,\ldots,N
$$

$$
t = 1,\ldots,T
$$

其中本文设定：

$$
N = 500,\quad T = 10
$$

每个个体具有两个可观测协变量：

$$
X_{1i} \sim N(0,1)
$$

$$
X_{2i} \sim B(0.5)
$$

其中 $X_{1i}$ 是连续变量，$X_{2i}$ 是离散变量。

此外，我们还生成两个不可观测个体特征：

$$
\alpha_i \sim N(0,1)
$$

$$
\eta_i \sim N(0,1)
$$

其中 $\alpha_i$ 表示个体固定效应，$\eta_i$ 用于在场景 C 中制造不可观测时变混淆。

### 1.1 处理分配机制

本文设置三个处理批次和一个 never-treated 对照组：

| 处理批次 $G_i$ | 含义 |
|---|---|
| $G_i = 4$ | 第 4 期开始接受处理 |
| $G_i = 6$ | 第 6 期开始接受处理 |
| $G_i = 8$ | 第 8 期开始接受处理 |
| $G_i = 0$ | never-treated，永远不接受处理 |

处理状态定义为：

$$
D_{it}=1(G_i>0,\ t\geq G_i)
$$

也就是说，如果某个个体在第 $G_i$ 期开始接受处理，那么在 $t \geq G_i$ 的时期中，$D_{it}=1$；否则 $D_{it}=0$。

相对处理时间定义为：

$$
rel\_time_{it}=t-G_i
$$

其中 $rel\_time=0$ 表示处理发生当期，$rel\_time<0$ 表示处理前，$rel\_time>0$ 表示处理后。

### 1.2 潜在结果和观测结果

未处理潜在结果设定为：

$$
Y_{it}(0)
=
2+\alpha_i+\lambda_t+0.5X_{1i}+0.8X_{2i}
+\text{TrendHeterogeneity}_{it}
+\varepsilon_{it}
$$

其中：

$$
\lambda_t = 0.2t
$$

表示共同时间趋势。

处理后潜在结果为：

$$
Y_{it}(1)=Y_{it}(0)+\tau_{it}
$$

最终观测结果为：

$$
Y_{it}
=
Y_{it}(0)+D_{it}\tau_{it}
$$

因此，如果个体尚未接受处理，则观测到 $Y_{it}(0)$；如果个体已经接受处理，则观测到 $Y_{it}(1)$。

### 1.3 四个 DGP 场景

本文设计四个 DGP 场景，用于考察不同估计方法在不同数据环境中的表现。

| 场景 | 设定 | 目的 |
|---|---|---|
| 场景 A | 随机处理分配；平行趋势成立；处理效应同质 | 基准情形 |
| 场景 B | 处理分配与可观测协变量 $X_1$、$X_2$ 相关；条件平行趋势成立 | 考察匹配方法对可观测选择偏误的修正能力 |
| 场景 C | 处理分配与不可观测因素 $\eta_i$ 相关，且 $\eta_i$ 影响未处理趋势 | 考察不可观测时变混淆下各方法的局限 |
| 场景 D | 随机处理分配；处理效应动态衰减且存在 cohort 异质性 | 考察动态处理效应和 cohort 异质性下 TWFE 的偏误 |

其中，场景 A、B、C 的处理效应设为同质效应：

$$
\tau_{it}=2
$$

场景 D 的处理效应设为动态且异质：

$$
\tau_{it}=\gamma_g \exp[-0.25(t-G_i)]
$$
其中：

| cohort | 初始处理效应 $\gamma_g$ |
|---|---|
| $G_i=4$ | 3.0 |
| $G_i=6$ | 2.0 |
| $G_i=8$ | 1.2 |

In [4]:
****************************************************
* Part 1.4：定义 DGP 生成程序
****************************************************

capture program drop generate_dgp

program define generate_dgp, rclass

    ************************************************
    * 1. 读取全局参数
    ************************************************

    local N = $N
    local T = $T

    ************************************************
    * 2. 生成个体层面数据
    ************************************************

    clear
    set obs `N'

    * 生成个体编号
    gen id = _n

    * 生成连续型协变量 x1
    gen x1 = rnormal(0, 1)

    * 生成离散型协变量 x2
    gen x2 = runiform() > 0.5

    * 生成个体固定效应
    gen alpha_i = rnormal(0, 1)

    * 生成不可观测个体特征
    gen eta_i = rnormal(0, 1)

    ************************************************
    * 3. 生成四个场景下的处理批次
    ************************************************

    ************************************************
    * 场景 A：随机处理分配
    ************************************************

    gen rand_A = runiform()
    sort rand_A

    gen g_A = .
    replace g_A = 4 if _n <= `N' * 0.25
    replace g_A = 6 if _n >  `N' * 0.25 & _n <= `N' * 0.50
    replace g_A = 8 if _n >  `N' * 0.50 & _n <= `N' * 0.75
    replace g_A = 0 if _n >  `N' * 0.75

    ************************************************
    * 场景 B：处理分配与可观测协变量相关
    ************************************************

    * x1 越大、x2=1 的个体越可能更早进入处理
    gen score_B = 0.8 * x1 + 0.8 * x2 + rnormal(0, 0.5)

    gsort -score_B

    gen g_B = .
    replace g_B = 4 if _n <= `N' * 0.25
    replace g_B = 6 if _n >  `N' * 0.25 & _n <= `N' * 0.50
    replace g_B = 8 if _n >  `N' * 0.50 & _n <= `N' * 0.75
    replace g_B = 0 if _n >  `N' * 0.75

    ************************************************
    * 场景 C：处理分配与不可观测因素相关
    ************************************************

    * eta_i 同时影响处理分配和未处理结果趋势
    gen score_C = 0.4 * x1 + 0.4 * x2 + 1.2 * eta_i + rnormal(0, 0.5)

    gsort -score_C

    gen g_C = .
    replace g_C = 4 if _n <= `N' * 0.25
    replace g_C = 6 if _n >  `N' * 0.25 & _n <= `N' * 0.50
    replace g_C = 8 if _n >  `N' * 0.50 & _n <= `N' * 0.75
    replace g_C = 0 if _n >  `N' * 0.75

    ************************************************
    * 场景 D：随机处理分配
    ************************************************

    gen rand_D = runiform()
    sort rand_D

    gen g_D = .
    replace g_D = 4 if _n <= `N' * 0.25
    replace g_D = 6 if _n >  `N' * 0.25 & _n <= `N' * 0.50
    replace g_D = 8 if _n >  `N' * 0.50 & _n <= `N' * 0.75
    replace g_D = 0 if _n >  `N' * 0.75

    * 恢复按照 id 排序
    sort id

    ************************************************
    * 4. 扩展为平衡面板数据
    ************************************************

    * 每个个体复制 T 次
    expand `T'

    * 在每个个体内部生成时间变量 t
    bysort id: gen t = _n

    * 按 id 和 t 排序
    sort id t

    ************************************************
    * 5. 生成处理状态和相对处理时间
    ************************************************

    foreach sc in A B C D {

        * D_it = 1(G_i > 0, t >= G_i)
        gen D_`sc' = (g_`sc' > 0 & t >= g_`sc')

        * 相对处理时间 rel_time = t - G_i
        gen rel_`sc' = .
        replace rel_`sc' = t - g_`sc' if g_`sc' > 0
    }

    ************************************************
    * 6. 生成未处理潜在结果 Y(0)
    ************************************************

    * 共同时间趋势
    gen lambda_t = 0.2 * t

    * 四个场景分别生成随机扰动项
    gen eps_A = rnormal(0, 1)
    gen eps_B = rnormal(0, 1)
    gen eps_C = rnormal(0, 1)
    gen eps_D = rnormal(0, 1)

    * 四个场景的趋势异质性
    gen trend_A = 0
    gen trend_B = (0.08 * x1 + 0.12 * x2) * t
    gen trend_C = (0.08 * x1 + 0.12 * x2) * t + 0.25 * eta_i * t
    gen trend_D = 0

    * 场景 A 的未处理潜在结果
    gen y0_A = 2 + alpha_i + lambda_t + 0.5 * x1 + 0.8 * x2 + trend_A + eps_A

    * 场景 B 的未处理潜在结果
    gen y0_B = 2 + alpha_i + lambda_t + 0.5 * x1 + 0.8 * x2 + trend_B + eps_B

    * 场景 C 的未处理潜在结果
    gen y0_C = 2 + alpha_i + lambda_t + 0.5 * x1 + 0.8 * x2 + trend_C + eps_C

    * 场景 D 的未处理潜在结果
    gen y0_D = 2 + alpha_i + lambda_t + 0.5 * x1 + 0.8 * x2 + trend_D + eps_D

    ************************************************
    * 7. 生成真实处理效应 tau
    ************************************************

    * 场景 A、B、C：同质处理效应
    gen tau_A = 0
    replace tau_A = 2 if D_A == 1

    gen tau_B = 0
    replace tau_B = 2 if D_B == 1

    gen tau_C = 0
    replace tau_C = 2 if D_C == 1

    * 场景 D：动态且 cohort 异质的处理效应
    gen tau_D = 0
    replace tau_D = 3.0 * exp(-0.25 * rel_D) if D_D == 1 & g_D == 4
    replace tau_D = 2.0 * exp(-0.25 * rel_D) if D_D == 1 & g_D == 6
    replace tau_D = 1.2 * exp(-0.25 * rel_D) if D_D == 1 & g_D == 8

    ************************************************
    * 8. 生成处理后潜在结果和最终观测结果
    ************************************************

    foreach sc in A B C D {

        * 处理后潜在结果 Y(1)
        gen y1_`sc' = y0_`sc' + tau_`sc'

        * 最终观测结果 Y = Y(0) + D * tau
        gen y_`sc' = y0_`sc' + D_`sc' * tau_`sc'
    }

    ************************************************
    * 9. 声明面板结构
    ************************************************

    xtset id t

    ************************************************
    * 10. 返回四个场景的真实 ATT
    ************************************************

    foreach sc in A B C D {

        quietly summarize tau_`sc' if D_`sc' == 1
        return scalar true_att_`sc' = r(mean)
    }

end

### 1.5 生成一份示例 DGP 并检查

在正式进行 Monte Carlo 模拟之前，先调用一次 `generate_dgp` 程序，生成一份示例数据。

本步骤主要检查：

1. 数据规模是否为 $500 \times 10 = 5000$；
2. 每个场景中是否都有四组个体：never-treated、第 4 期处理、第 6 期处理、第 8 期处理；
3. 每一期的处理比例是否符合交错处理结构；
4. 四个场景的真实 ATT 是否符合预期。

In [5]:
****************************************************
* Part 1.5：生成一份示例 DGP 并检查
****************************************************

* 调用 DGP 程序生成一份示例数据
generate_dgp

****************************************************
* 1. 检查数据规模
****************************************************

display "=============================================="
display "数据规模检查"
display "=============================================="

count

display "理论总观测数 N*T = " $N * $T

****************************************************
* 2. 检查 cohort 分布
****************************************************

capture matrix drop cohort_check

matrix cohort_check = J(4, 4, .)

matrix colnames cohort_check = G0 G4 G6 G8

matrix rownames cohort_check = Scenario_A Scenario_B Scenario_C Scenario_D

local row = 1

foreach sc in A B C D {

    quietly count if t == 1 & g_`sc' == 0
    matrix cohort_check[`row',1] = r(N)

    quietly count if t == 1 & g_`sc' == 4
    matrix cohort_check[`row',2] = r(N)

    quietly count if t == 1 & g_`sc' == 6
    matrix cohort_check[`row',3] = r(N)

    quietly count if t == 1 & g_`sc' == 8
    matrix cohort_check[`row',4] = r(N)

    local row = `row' + 1
}

display "cohort 分布检查："
matrix list cohort_check, format(%9.0f)

****************************************************
* 3. 检查真实 ATT
****************************************************

capture matrix drop true_att_check

matrix true_att_check = J(4, 1, .)

matrix colnames true_att_check = True_ATT

matrix rownames true_att_check = Scenario_A Scenario_B Scenario_C Scenario_D

local row = 1

foreach sc in A B C D {

    quietly summarize tau_`sc' if D_`sc' == 1
    matrix true_att_check[`row',1] = r(mean)

    local row = `row' + 1
}

display "真实 ATT 检查："
matrix list true_att_check, format(%9.3f)

****************************************************
* 4. 检查处理比例随时间变化
****************************************************

preserve

collapse (mean) D_A D_B D_C D_D, by(t)

rename D_A Share_A
rename D_B Share_B
rename D_C Share_C
rename D_D Share_D

display "各时期处理比例："
list t Share_A Share_B Share_C Share_D, noobs

restore

****************************************************
* 5. 保存示例 DGP 数据
****************************************************

save "example_dgp.dta", replace


Number of observations (_N) was 0, now 500.
(500 missing values generated)
(125 real changes made)
(125 real changes made)
(125 real changes made)
(125 real changes made)
(500 missing values generated)
(125 real changes made)
(125 real changes made)
(125 real changes made)
(125 real changes made)
(500 missing values generated)
(125 real changes made)
(125 real changes made)
(125 real changes made)
(125 real changes made)
(500 missing values generated)
(125 real changes made)
(125 real changes made)
(125 real changes made)
(125 real changes made)
(4,500 observations created)
(5,000 missing values generated)
(3,750 real changes made)
(5,000 missing values generated)
(3,750 real changes made)
(5,000 missing values generated)
(3,750 real changes made)
(5,000 missing values generated)
(3,750 real changes made)
(1,875 real changes made)
(1,875 real changes made)
(1,875 real changes made)
(875 real changes made)
(625 real changes made)
(375 real changes made)

Panel variable: id (strongly ba

### 1.6 DGP 的 Monte Carlo 验证

在正式比较不同估计方法之前，我们先对 DGP 本身进行 Monte Carlo 验证。

本步骤不估计任何 DID 模型，只反复调用 `generate_dgp` 程序，检查每次生成的数据是否满足作业要求。

具体检查内容包括：

1. 每次生成的数据是否包含 $N \times T = 500 \times 10 = 5000$ 条观测；
2. 四个场景中是否都包含 never-treated 组和三个处理批次；
3. 每个 cohort 的样本量是否稳定；
4. 四个场景的真实 ATT 是否符合 DGP 设定。

由于本文正式设定 Monte Carlo 重复次数为：

$$
R = 1000
$$

因此本节将重复生成 1000 份 DGP 数据，并汇总真实 ATT 和 cohort 分布。

In [6]:
****************************************************
* Part 1.6：DGP 的 Monte Carlo 验证
****************************************************

****************************************************
* 1. 定义单次 DGP 验证程序
****************************************************

capture program drop validate_dgp_once

program define validate_dgp_once, rclass

    ************************************************
    * 1.1 调用 DGP 程序，生成一份新的模拟数据
    ************************************************

    quietly generate_dgp

    ************************************************
    * 1.2 返回数据规模
    ************************************************

    * 当前数据总观测数
    return scalar nobs = _N

    * 当前个体数量
    * 因为数据是平衡面板，每个 id 有 T 期，所以个体数量 = 总观测数 / T
    quietly count
    return scalar n_id = r(N) / $T

    ************************************************
    * 1.3 返回四个场景下的 cohort 分布
    ************************************************

    foreach sc in A B C D {

        * never-treated 组人数
        quietly count if t == 1 & g_`sc' == 0
        return scalar n_g0_`sc' = r(N)

        * 第 4 期处理组人数
        quietly count if t == 1 & g_`sc' == 4
        return scalar n_g4_`sc' = r(N)

        * 第 6 期处理组人数
        quietly count if t == 1 & g_`sc' == 6
        return scalar n_g6_`sc' = r(N)

        * 第 8 期处理组人数
        quietly count if t == 1 & g_`sc' == 8
        return scalar n_g8_`sc' = r(N)
    }

    ************************************************
    * 1.4 返回四个场景下的真实 ATT
    ************************************************

    foreach sc in A B C D {

        quietly summarize tau_`sc' if D_`sc' == 1
        return scalar true_att_`sc' = r(mean)
    }

end

****************************************************
* 2. 运行 DGP Monte Carlo 验证
****************************************************

simulate ///
    nobs = r(nobs) ///
    n_id = r(n_id) ///
    true_att_A = r(true_att_A) ///
    true_att_B = r(true_att_B) ///
    true_att_C = r(true_att_C) ///
    true_att_D = r(true_att_D) ///
    n_g0_A = r(n_g0_A) n_g4_A = r(n_g4_A) n_g6_A = r(n_g6_A) n_g8_A = r(n_g8_A) ///
    n_g0_B = r(n_g0_B) n_g4_B = r(n_g4_B) n_g6_B = r(n_g6_B) n_g8_B = r(n_g8_B) ///
    n_g0_C = r(n_g0_C) n_g4_C = r(n_g4_C) n_g6_C = r(n_g6_C) n_g8_C = r(n_g8_C) ///
    n_g0_D = r(n_g0_D) n_g4_D = r(n_g4_D) n_g6_D = r(n_g6_D) n_g8_D = r(n_g8_D), ///
    reps($R) seed(20260529) dots(100): validate_dgp_once

****************************************************
* 3. 汇总数据规模检查
****************************************************

display "=============================================="
display "DGP Monte Carlo：数据规模检查"
display "=============================================="

summarize nobs n_id

****************************************************
* 4. 汇总真实 ATT
****************************************************

capture matrix drop dgp_att_summary

matrix dgp_att_summary = J(4, 4, .)

matrix colnames dgp_att_summary = Mean SD Min Max

matrix rownames dgp_att_summary = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

local row = 1

foreach sc in A B C D {

    quietly summarize true_att_`sc'

    matrix dgp_att_summary[`row',1] = r(mean)
    matrix dgp_att_summary[`row',2] = r(sd)
    matrix dgp_att_summary[`row',3] = r(min)
    matrix dgp_att_summary[`row',4] = r(max)

    local row = `row' + 1
}

display "真实 ATT 的 Monte Carlo 汇总："
matrix list dgp_att_summary, format(%9.3f)

****************************************************
* 5. 汇总 cohort 分布
****************************************************

capture matrix drop dgp_cohort_summary

matrix dgp_cohort_summary = J(4, 4, .)

matrix colnames dgp_cohort_summary = G0 G4 G6 G8

matrix rownames dgp_cohort_summary = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

local row = 1

foreach sc in A B C D {

    quietly summarize n_g0_`sc'
    matrix dgp_cohort_summary[`row',1] = r(mean)

    quietly summarize n_g4_`sc'
    matrix dgp_cohort_summary[`row',2] = r(mean)

    quietly summarize n_g6_`sc'
    matrix dgp_cohort_summary[`row',3] = r(mean)

    quietly summarize n_g8_`sc'
    matrix dgp_cohort_summary[`row',4] = r(mean)

    local row = `row' + 1
}

display "cohort 分布的 Monte Carlo 汇总："
matrix list dgp_cohort_summary, format(%9.0f)

****************************************************
* 6. 保存 DGP Monte Carlo 验证结果
****************************************************

save "dgp_mc_validation_results.dta", replace

****************************************************
* 7. 重新生成一份示例 DGP，供后续估计方法展示使用
****************************************************

generate_dgp

save "example_dgp.dta", replace





      Command: validate_dgp_once
         nobs: r(nobs)
         n_id: r(n_id)
   true_att_A: r(true_att_A)
   true_att_B: r(true_att_B)
   true_att_C: r(true_att_C)
   true_att_D: r(true_att_D)
       n_g0_A: r(n_g0_A)
       n_g4_A: r(n_g4_A)
       n_g6_A: r(n_g6_A)
       n_g8_A: r(n_g8_A)
       n_g0_B: r(n_g0_B)
       n_g4_B: r(n_g4_B)
       n_g6_B: r(n_g6_B)
       n_g8_B: r(n_g8_B)
       n_g0_C: r(n_g0_C)
       n_g4_C: r(n_g4_C)
       n_g6_C: r(n_g6_C)
       n_g8_C: r(n_g8_C)
       n_g0_D: r(n_g0_D)
       n_g4_D: r(n_g4_D)
       n_g6_D: r(n_g6_D)
       n_g8_D: r(n_g8_D)

Simulations (1,000): .........1,000 done


DGP Monte Carlo：数据规模检查



    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
        nobs |      1,000        5000           0       5000       5000
        n_id |      1,000         500           0        500        500







真实 ATT 的 Monte Carlo 汇总：


dgp_att_s

## Part 2. Estimation Methods

在 Part 1 中，我们已经完成了 DGP 设计，并通过 Monte Carlo 验证了 DGP 的样本结构和真实处理效应。

本部分开始实现不同估计方法。

首先，我们使用传统的双向固定效应模型，即 TWFE：

$$
Y_{it} = \alpha_i + \lambda_t + \beta D_{it} + \varepsilon_{it}
$$

其中：

- $\alpha_i$ 表示个体固定效应；
- $\lambda_t$ 表示时间固定效应；
- $D_{it}$ 表示个体 $i$ 在时期 $t$ 是否已经接受处理；
- $\beta$ 是 TWFE 估计得到的平均处理效应。

在本节中，我们分别对四个场景 A、B、C、D 估计 TWFE，并比较 TWFE 估计值和 DGP 中的真实 ATT。

需要注意的是，TWFE 是传统 DID 中最常见的基准方法，但在交错处理设定下，如果存在动态处理效应、cohort 异质性或非平行趋势，TWFE 可能产生系统性偏误。

In [7]:
****************************************************
* Part 2.1：TWFE 双向固定效应估计
****************************************************

****************************************************
* 1. 读取示例 DGP 数据
****************************************************

* 读取 Part 1 中保存的示例 DGP 数据
use "example_dgp.dta", clear

* 声明面板结构
quietly xtset id t

****************************************************
* 2. 创建 TWFE 结果矩阵
****************************************************

* 删除可能已经存在的旧矩阵
capture matrix drop twfe_example_table

* 4 行对应四个场景 A/B/C/D
* 4 列分别为：真实 ATT、TWFE 估计值、标准误、单次样本偏误
matrix twfe_example_table = J(4, 4, .)

matrix colnames twfe_example_table = True_ATT TWFE_Beta SE Bias

matrix rownames twfe_example_table = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

****************************************************
* 3. 对四个场景分别运行 TWFE
****************************************************

local row = 1

foreach sc in A B C D {

    ************************************************
    * 3.1 计算当前场景的真实 ATT
    ************************************************

    * 真实 ATT 定义为：所有已处理观测的真实 tau 平均值
    quietly summarize tau_`sc' if D_`sc' == 1
    local true_att = r(mean)

    ************************************************
    * 3.2 运行 TWFE 回归
    ************************************************

    * y_`sc' 是当前场景的结果变量
    * D_`sc' 是当前场景的处理状态变量
    * i.t 控制时间固定效应
    * fe 控制个体固定效应
    * vce(cluster id) 将标准误聚类到个体层面
    quietly xtreg y_`sc' D_`sc' i.t, fe vce(cluster id)

    ************************************************
    * 3.3 提取估计结果
    ************************************************

    * 提取 TWFE 估计系数
    local beta = _b[D_`sc']

    * 提取聚类稳健标准误
    local se = _se[D_`sc']

    * 计算单次样本偏误
    local bias = `beta' - `true_att'

    ************************************************
    * 3.4 将结果填入矩阵
    ************************************************

    matrix twfe_example_table[`row', 1] = `true_att'
    matrix twfe_example_table[`row', 2] = `beta'
    matrix twfe_example_table[`row', 3] = `se'
    matrix twfe_example_table[`row', 4] = `bias'

    local row = `row' + 1
}

****************************************************
* 4. 输出 TWFE 示例结果
****************************************************

display "TWFE 示例估计结果："
matrix list twfe_example_table, format(%9.3f)










TWFE 示例估计结果：


twfe_example_table[4,4]
             True_ATT  TWFE_Beta         SE       Bias
Scenario_A      2.000      1.909      0.053     -0.091
Scenario_B      2.000      2.418      0.056      0.418
Scenario_C      2.000      3.231      0.095      1.231
Scenario_D      1.368      1.793      0.056      0.425


### 2.2 同期 PSM-DID 估计

本节实现同期 PSM-DID。

同期 PSM-DID 的基本思想是：对于每一个处理批次 cohort，只比较处理前一期和处理发生当期的结果变化。

对于处理批次 $G_i=g$，定义：

$$
\Delta Y_i = Y_{i,g} - Y_{i,g-1}
$$

其中 $g-1$ 是处理前一期，$g$ 是处理发生当期。

在每一个 cohort 中：

- 处理组：$G_i=g$ 的个体；
- 控制组：在时期 $g$ 尚未接受处理的个体，包括 never-treated 个体和未来才接受处理的个体；
- 匹配变量：$X_1$ 和 $X_2$；
- 估计对象：当前 cohort 在处理发生当期的 ATT。

最后，将三个 cohort 的 ATT 按处理组样本量加权平均，得到当前场景下的同期 PSM-DID 估计结果。

需要注意的是，同期 PSM-DID 估计的是“处理刚发生时”的效应，因此在场景 D 中，它对应的是初始处理效应，而不是所有处理后时期上的平均 ATT。

In [8]:
****************************************************
* Part 2.2：同期 PSM-DID 估计
****************************************************

****************************************************
* 1. 读取示例 DGP 数据
****************************************************

use "example_dgp.dta", clear

quietly xtset id t

* 保存一份完整示例数据，后面每次循环都从这里重新读取
tempfile base
save `base', replace

****************************************************
* 2. 创建同期 PSM-DID 结果矩阵
****************************************************

capture matrix drop psmdid_example_table

* 4 行对应场景 A/B/C/D
* 4 列分别为：真实 ATT、PSM-DID 估计值、标准误、单次样本偏误
matrix psmdid_example_table = J(4, 4, .)

matrix colnames psmdid_example_table = True_ATT PSM_DID SE Bias

matrix rownames psmdid_example_table = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

****************************************************
* 3. 对四个场景分别运行同期 PSM-DID
****************************************************

local row = 1

foreach sc in A B C D {

    ************************************************
    * 3.1 初始化当前场景的加权汇总变量
    ************************************************

    local sum_n = 0
    local sum_true = 0
    local sum_est = 0
    local sum_se2 = 0

    ************************************************
    * 3.2 对三个处理批次分别做同期 PSM-DID
    ************************************************

    foreach cohort in 4 6 8 {

        ************************************************
        * 3.2.1 读取完整示例数据
        ************************************************

        use `base', clear

        * 当前 cohort 的处理前一期
        local pre = `cohort' - 1

        ************************************************
        * 3.2.2 只保留处理前一期和处理当期
        ************************************************

        keep if inlist(t, `pre', `cohort')

        ************************************************
        * 3.2.3 保留当前 cohort 的处理组和 clean controls
        ************************************************

        * 处理组：g = 当前 cohort
        * 控制组：g = 0 的 never-treated，或者 g > cohort 的未来处理组
        * g < cohort 的早处理组已经接受处理，不能作为 clean controls
        keep if g_`sc' == `cohort' | g_`sc' == 0 | g_`sc' > `cohort'

        ************************************************
        * 3.2.4 定义当前 cohort 的处理组变量
        ************************************************

        gen treat = (g_`sc' == `cohort')

        ************************************************
        * 3.2.5 构造处理前后变化量 dy
        ************************************************

        * 处理前一期结果
        gen y_pre_tmp = y_`sc' if t == `pre'

        * 处理当期结果
        gen y_post_tmp = y_`sc' if t == `cohort'

        * 将同一个 id 的处理前结果和处理当期结果整理到同一行
        bysort id: egen y_pre = max(y_pre_tmp)
        bysort id: egen y_post = max(y_post_tmp)

        * 结果变化量
        gen dy = y_post - y_pre

        * 每个个体只保留处理当期这一行
        keep if t == `cohort'

        * 删除缺失的变化量
        drop if missing(dy)

        ************************************************
        * 3.2.6 计算当前 cohort 的真实 ATT
        ************************************************

        * 同期 PSM-DID 的真实 ATT 是处理刚发生当期的真实 tau
        quietly summarize tau_`sc' if treat == 1
        local true_g = r(mean)

        * 当前 cohort 的处理组样本量
        quietly count if treat == 1
        local n_g = r(N)

        ************************************************
        * 3.2.7 使用倾向得分匹配估计 ATT
        ************************************************

        * 结果变量：dy
        * 处理变量：treat
        * 匹配变量：x1 和 x2
        * atet 表示估计处理组平均处理效应
        quietly teffects psmatch (dy) (treat x1 x2), atet

        * 提取估计值和标准误
        matrix b = e(b)
        matrix V = e(V)

        local att_g = b[1,1]
        local se_g = sqrt(V[1,1])

        ************************************************
        * 3.2.8 按处理组样本量加权汇总
        ************************************************

        local sum_n = `sum_n' + `n_g'
        local sum_true = `sum_true' + `n_g' * `true_g'
        local sum_est = `sum_est' + `n_g' * `att_g'
        local sum_se2 = `sum_se2' + (`n_g'^2) * (`se_g'^2)
    }

    ************************************************
    * 3.3 汇总当前场景三个 cohort 的结果
    ************************************************

    local true_avg = `sum_true' / `sum_n'
    local est_avg = `sum_est' / `sum_n'

    * 这里的标准误是对三个 cohort 标准误的简化加权汇总
    local se_avg = sqrt(`sum_se2') / `sum_n'

    local bias_avg = `est_avg' - `true_avg'

    ************************************************
    * 3.4 将当前场景结果放入矩阵
    ************************************************

    matrix psmdid_example_table[`row', 1] = `true_avg'
    matrix psmdid_example_table[`row', 2] = `est_avg'
    matrix psmdid_example_table[`row', 3] = `se_avg'
    matrix psmdid_example_table[`row', 4] = `bias_avg'

    local row = `row' + 1
}

****************************************************
* 4. 输出同期 PSM-DID 示例结果
****************************************************

display "同期 PSM-DID 示例估计结果："
matrix list psmdid_example_table, format(%9.3f)

****************************************************
* 5. 恢复示例 DGP 数据，方便后续继续使用
****************************************************

use `base', clear

quietly xtset id t





(file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000002.tmp not found)
file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000002.tmp saved as .dta
    format






(4,000 observations deleted)
(0 observations deleted)
(500 missing values generated)
(500 missing values generated)
(500 observations deleted)
(0 observations deleted)
(4,000 observations deleted)
(250 observations deleted)
(375 missing values generated)
(375 missing values generated)
(375 observations deleted)
(0 observations deleted)
(4,000 observations deleted)
(500 observations deleted)
(250 missing values generated)
(250 missing values generated)
(250 observations deleted)
(0 observations deleted)
(4,000 observations deleted)
(0 observations deleted)
(500 missing values generated)
(500 missing values generated)
(500 observations deleted)
(0 observations deleted)
(4,000 observations deleted)
(250 observations deleted)
(375 missing values generated)
(375 missing values generated)
(375 observations deleted)
(0 observa

### 2.3 滚动 PSM-DID 估计

本节实现滚动 PSM-DID。

同期 PSM-DID 只比较处理前一期和处理发生当期，而滚动 PSM-DID 会对每一个处理后时期分别构造 DID 比较。

对于处理批次 $G_i=g$，固定处理前一期为 $g-1$。对于每一个处理后时期 $t \geq g$，定义：

$$
\Delta Y_{it}=Y_{it}-Y_{i,g-1}
$$

其中：

- 处理组是 $G_i=g$ 的个体；
- 控制组是在时期 $t$ 仍然没有接受处理的 clean controls；
- clean controls 包括 never-treated 个体和在 $t$ 之后才接受处理的个体。

因此，对于每一个 $(g,t)$ 单元，我们重新选择可用控制组，并使用 $X_1$ 和 $X_2$ 进行倾向得分匹配。

最后，将所有 $(g,t)$ 单元的 ATT 按处理组样本量加权平均，得到当前场景下的滚动 PSM-DID 估计结果。

相比同期 PSM-DID，滚动 PSM-DID 使用了更多处理后时期，因此其目标参数更接近所有处理后时期上的平均 ATT。

In [9]:
****************************************************
* Part 2.3：滚动 PSM-DID 估计
****************************************************

****************************************************
* 1. 读取示例 DGP 数据
****************************************************

use "example_dgp.dta", clear

quietly xtset id t

* 保存一份完整示例数据，后面每次循环都从这里重新读取
tempfile base
save `base', replace

****************************************************
* 2. 创建滚动 PSM-DID 结果矩阵
****************************************************

capture matrix drop rolling_psm_example_table

* 4 行对应场景 A/B/C/D
* 4 列分别为：真实 ATT、滚动 PSM-DID 估计值、标准误、单次样本偏误
matrix rolling_psm_example_table = J(4, 4, .)

matrix colnames rolling_psm_example_table = True_ATT Rolling_PSM_DID SE Bias

matrix rownames rolling_psm_example_table = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

****************************************************
* 3. 对四个场景分别运行滚动 PSM-DID
****************************************************

local row = 1

foreach sc in A B C D {

    ************************************************
    * 3.1 初始化当前场景的加权汇总变量
    ************************************************

    local sum_n = 0
    local sum_true = 0
    local sum_est = 0
    local sum_se2 = 0

    ************************************************
    * 3.2 对三个处理批次分别循环
    ************************************************

    foreach cohort in 4 6 8 {

        * 当前 cohort 的处理前一期
        local pre = `cohort' - 1

        ************************************************
        * 3.3 对当前 cohort 的所有处理后时期循环
        ************************************************

        forvalues post = `cohort'/$T {

            ************************************************
            * 3.3.1 读取完整示例数据
            ************************************************

            use `base', clear

            ************************************************
            * 3.3.2 只保留处理前一期和当前处理后时期
            ************************************************

            keep if t == `pre' | t == `post'

            ************************************************
            * 3.3.3 保留当前 cohort 的处理组和 clean controls
            ************************************************

            * 处理组：g = 当前 cohort
            * 控制组：g = 0 的 never-treated，或者 g > post 的未来处理组
            * 注意：如果某个未来处理组在当前 post 期已经接受处理，则不能作为控制组
            keep if g_`sc' == `cohort' | g_`sc' == 0 | g_`sc' > `post'

            ************************************************
            * 3.3.4 定义当前 cohort 的处理组变量
            ************************************************

            gen treat = (g_`sc' == `cohort')

            ************************************************
            * 3.3.5 构造从处理前一期到当前 post 期的变化量 dy
            ************************************************

            * 处理前一期结果
            gen y_pre_tmp = y_`sc' if t == `pre'

            * 当前处理后时期结果
            gen y_post_tmp = y_`sc' if t == `post'

            * 将同一个 id 的处理前结果和 post 期结果整理到同一行
            bysort id: egen y_pre = max(y_pre_tmp)
            bysort id: egen y_post = max(y_post_tmp)

            * 结果变化量
            gen dy = y_post - y_pre

            * 每个个体只保留当前 post 期这一行
            keep if t == `post'

            * 删除缺失的变化量
            drop if missing(dy)

            ************************************************
            * 3.3.6 计算当前 (cohort, post) 单元的真实 ATT
            ************************************************

            * 当前单元的真实 ATT 是处理组在当前 post 期的真实 tau 均值
            quietly summarize tau_`sc' if treat == 1
            local true_gt = r(mean)

            * 当前单元处理组样本量
            quietly count if treat == 1
            local n_gt = r(N)

            ************************************************
            * 3.3.7 使用倾向得分匹配估计 ATT
            ************************************************

            * 结果变量：dy
            * 处理变量：treat
            * 匹配变量：x1 和 x2
            * atet 表示估计处理组平均处理效应
            quietly teffects psmatch (dy) (treat x1 x2), atet pstolerance(1e-10)

            * 提取估计值和标准误
            matrix b = e(b)
            matrix V = e(V)

            local att_gt = b[1,1]
            local se_gt = sqrt(V[1,1])

            ************************************************
            * 3.3.8 按处理组样本量加权汇总
            ************************************************

            local sum_n = `sum_n' + `n_gt'
            local sum_true = `sum_true' + `n_gt' * `true_gt'
            local sum_est = `sum_est' + `n_gt' * `att_gt'
            local sum_se2 = `sum_se2' + (`n_gt'^2) * (`se_gt'^2)
        }
    }

    ************************************************
    * 3.4 汇总当前场景所有 (cohort, post) 单元的结果
    ************************************************

    local true_avg = `sum_true' / `sum_n'
    local est_avg = `sum_est' / `sum_n'

    * 这里的标准误是对多个 (cohort, post) 单元标准误的简化加权汇总
    local se_avg = sqrt(`sum_se2') / `sum_n'

    local bias_avg = `est_avg' - `true_avg'

    ************************************************
    * 3.5 将当前场景结果放入矩阵
    ************************************************

    matrix rolling_psm_example_table[`row', 1] = `true_avg'
    matrix rolling_psm_example_table[`row', 2] = `est_avg'
    matrix rolling_psm_example_table[`row', 3] = `se_avg'
    matrix rolling_psm_example_table[`row', 4] = `bias_avg'

    local row = `row' + 1
}

****************************************************
* 4. 输出滚动 PSM-DID 示例结果
****************************************************

display "滚动 PSM-DID 示例估计结果："
matrix list rolling_psm_example_table, format(%9.3f)

****************************************************
* 5. 恢复示例 DGP 数据，方便后续继续使用
****************************************************

use `base', clear

quietly xtset id t





(file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000003.tmp not found)
file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000003.tmp saved as .dta
    format






(4,000 observations deleted)
(0 observations deleted)
(500 missing values generated)
(500 missing values generated)
(500 observations deleted)
(0 observations deleted)
(4,000 observations deleted)
(0 observations deleted)
(500 missing values generated)
(500 missing values generated)
(500 observations deleted)
(0 observations deleted)
(4,000 observations deleted)
(250 observations deleted)
(375 missing values generated)
(375 missing values generated)
(375 observations deleted)
(0 observations deleted)
(4,000 observations deleted)
(250 observations deleted)
(375 missing values generated)
(375 missing values generated)
(375 observations deleted)
(0 observations deleted)
(4,000 observations deleted)
(500 observations deleted)
(250 missing values generated)
(250 missing values generated)
(250 observations deleted)
(0 observa

### 2.4 合成控制法 SCM

本节实现合成控制法，也就是 Synthetic Control Method, SCM。

在交错处理设定下，为了避免已经接受处理的个体污染控制组，本节只使用 never-treated 个体作为 donor pool。

对于每一个场景和每一个处理批次 cohort，我们执行以下步骤：

1. 将当前 cohort 的处理组个体按时期取平均，构造一个“聚合处理单位”；
2. 使用 never-treated 个体作为 donor pool；
3. 利用处理前结果路径拟合合成控制组；
4. 计算处理后时期中，处理组与合成控制组之间的平均 gap；
5. 将该平均 gap 作为 SCM 估计的 ATT。

对于每个场景，我们分别对 $G_i=4$、$G_i=6$、$G_i=8$ 三个 cohort 进行 SCM 估计，然后按照处理后观测数进行加权平均。

In [10]:
****************************************************
* Part 2.4：合成控制法 SCM
****************************************************

****************************************************
* 1. 读取示例 DGP 数据
****************************************************

use "example_dgp.dta", clear

quietly xtset id t

* 保存一份完整示例数据
* 后面每次构造 donor pool 或处理组时，都从这里重新读取
tempfile base donors treated
save `base', replace

****************************************************
* 2. 创建 cohort 层面结果矩阵
****************************************************

capture matrix drop scm_cohort_example_table

* 12 行 = 4 个场景 × 3 个 cohort
* 5 列分别为：真实 ATT、SCM 估计值、偏误、处理后观测数、处理前时期数
matrix scm_cohort_example_table = J(12, 5, .)

matrix colnames scm_cohort_example_table = True_ATT SCM_ATT Bias N_Post Pre_Periods

matrix rownames scm_cohort_example_table = ///
    A_g4 A_g6 A_g8 ///
    B_g4 B_g6 B_g8 ///
    C_g4 C_g6 C_g8 ///
    D_g4 D_g6 D_g8

****************************************************
* 3. 创建场景汇总结果矩阵
****************************************************

capture matrix drop scm_summary_example_table

* 4 行对应场景 A/B/C/D
* 3 列分别为：加权真实 ATT、加权 SCM 估计值、加权偏误
matrix scm_summary_example_table = J(4, 3, .)

matrix colnames scm_summary_example_table = True_ATT SCM_ATT Bias

matrix rownames scm_summary_example_table = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

****************************************************
* 4. 设置矩阵行号
****************************************************

local cohort_row = 1
local scenario_row = 1

****************************************************
* 5. 对四个场景分别运行 SCM
****************************************************

foreach sc in A B C D {

    ************************************************
    * 5.1 初始化当前场景的加权汇总变量
    ************************************************

    local sum_n = 0
    local sum_true = 0
    local sum_scm = 0

    ************************************************
    * 5.2 对三个处理批次分别运行 SCM
    ************************************************

    foreach cohort in 4 6 8 {

        ************************************************
        * 5.2.1 读取完整示例数据
        ************************************************

        use `base', clear

        ************************************************
        * 5.2.2 设置当前 cohort 的基本参数
        ************************************************

        * 处理前最后一期
        local pre_end = `cohort' - 1

        * 处理前时期数量
        local pre_periods = `pre_end'

        * 给聚合处理单位设置一个不会和原始 id 冲突的编号
        if "`sc'" == "A" local base_id = 9000
        if "`sc'" == "B" local base_id = 9100
        if "`sc'" == "C" local base_id = 9200
        if "`sc'" == "D" local base_id = 9300

        local treated_unit = `base_id' + `cohort'

        ************************************************
        * 5.2.3 计算当前 cohort 的真实 ATT
        ************************************************

        * 真实 ATT：当前 cohort 所有处理后时期的真实 tau 平均值
        quietly summarize tau_`sc' if g_`sc' == `cohort' & t >= `cohort'
        local true_att = r(mean)

        * 当前 cohort 的处理后观测数
        quietly count if g_`sc' == `cohort' & t >= `cohort'
        local n_post = r(N)

        ************************************************
        * 5.2.4 构造处理前结果预测变量
        ************************************************

        * cohort=4 时，使用 y_scm(1) y_scm(2) y_scm(3)
        * cohort=6 时，使用 y_scm(1) 到 y_scm(5)
        * cohort=8 时，使用 y_scm(1) 到 y_scm(7)
        local predlist ""

        forvalues pp = 1/`pre_end' {
            local predlist "`predlist' y_scm(`pp')"
        }

        ************************************************
        * 5.2.5 构造 donor pool
        ************************************************

        use `base', clear

        * donor pool 只使用 never-treated 个体
        keep if g_`sc' == 0

        * 合成控制法中的单位编号
        gen synth_id = id

        * 统一结果变量名称
        gen y_scm = y_`sc'

        * 只保留合成控制需要的变量
        keep synth_id t y_scm

        * 保存 donor pool
        save `donors', replace

        ************************************************
        * 5.2.6 构造聚合处理单位
        ************************************************

        use `base', clear

        * 只保留当前 cohort 的处理组
        keep if g_`sc' == `cohort'

        * 将当前 cohort 的处理组在每一期的结果取平均
        collapse (mean) y_scm = y_`sc', by(t)

        * 给聚合处理单位赋予编号
        gen synth_id = `treated_unit'

        * 只保留合成控制需要的变量
        keep synth_id t y_scm

        * 保存聚合处理单位
        save `treated', replace

        ************************************************
        * 5.2.7 合并 donor pool 和聚合处理单位
        ************************************************

        use `donors', clear
        append using `treated'

        * 声明合成控制法所需的面板结构
        tsset synth_id t

        ************************************************
        * 5.2.8 执行合成控制法
        ************************************************

        * 当前 cohort 的 SCM 输出文件名
        local outfile "scm_`sc'_g`cohort'_output.dta"

        * 使用处理前结果路径拟合合成控制组
        synth y_scm ///
            `predlist', ///
            trunit(`treated_unit') ///
            trperiod(`cohort') ///
            keep("`outfile'") replace

        ************************************************
        * 5.2.9 读取 synth 输出并计算 SCM ATT
        ************************************************

        use "`outfile'", clear

        * gap 表示处理组结果与合成控制组结果之间的差距
        gen gap = _Y_treated - _Y_synthetic

        * SCM ATT：处理后时期 gap 的平均值
        quietly summarize gap if _time >= `cohort'
        local scm_att = r(mean)

        * 单次样本偏误
        local bias = `scm_att' - `true_att'

        ************************************************
        * 5.2.10 保存当前 cohort 结果
        ************************************************

        matrix scm_cohort_example_table[`cohort_row', 1] = `true_att'
        matrix scm_cohort_example_table[`cohort_row', 2] = `scm_att'
        matrix scm_cohort_example_table[`cohort_row', 3] = `bias'
        matrix scm_cohort_example_table[`cohort_row', 4] = `n_post'
        matrix scm_cohort_example_table[`cohort_row', 5] = `pre_periods'

        local cohort_row = `cohort_row' + 1

        ************************************************
        * 5.2.11 加入当前场景的加权汇总
        ************************************************

        local sum_n = `sum_n' + `n_post'
        local sum_true = `sum_true' + `n_post' * `true_att'
        local sum_scm = `sum_scm' + `n_post' * `scm_att'
    }

    ************************************************
    * 5.3 汇总当前场景的 SCM 结果
    ************************************************

    local true_avg = `sum_true' / `sum_n'
    local scm_avg = `sum_scm' / `sum_n'
    local bias_avg = `scm_avg' - `true_avg'

    matrix scm_summary_example_table[`scenario_row', 1] = `true_avg'
    matrix scm_summary_example_table[`scenario_row', 2] = `scm_avg'
    matrix scm_summary_example_table[`scenario_row', 3] = `bias_avg'

    local scenario_row = `scenario_row' + 1
}

****************************************************
* 6. 输出 cohort 层面 SCM 示例结果
****************************************************

display "SCM 示例估计结果：cohort 层面"
matrix list scm_cohort_example_table, format(%9.3f)

****************************************************
* 7. 输出场景汇总 SCM 示例结果
****************************************************

display "SCM 示例估计结果：场景汇总层面"
matrix list scm_summary_example_table, format(%9.3f)

****************************************************
* 8. 恢复示例 DGP 数据，方便后续继续使用
****************************************************

use `base', clear

quietly xtset id t





(file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000004.tmp not found)
file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000004.tmp saved as .dta
    format











(3,750 observations deleted)
(file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000005.tmp not found)
file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000005.tmp saved as .dta
    format
(3,750 observations deleted)
(file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000006.tmp not found)
file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000006.tmp saved as .dta
    format

Panel variable: synth_id (strongly balanced)
 Time variable: t, 1 to 10
         Delta: 1 unit
--------------------------------------------------------------------------------
Synthetic Control Method for Comparative Case Studies
--------------------------------------------------------------------------------

First Step: Data Setup
--------------------------------------------------------------------------------
---------------------------------

### 2.5 Callaway and Sant'Anna DID 估计

本节使用 Callaway and Sant'Anna DID 方法，也就是 Stata 中的 `csdid` 命令。

在交错处理设定下，传统 TWFE 可能会因为错误地使用 already-treated units 作为控制组而产生偏误。Callaway and Sant'Anna 方法的核心思想是先估计每个处理批次和每个时期的组别-时间平均处理效应：

$$
ATT(g,t)
$$

其中：

- $g$ 表示处理批次；
- $t$ 表示时期；
- $ATT(g,t)$ 表示第 $g$ 期处理组在时期 $t$ 的平均处理效应。

然后再将不同 $ATT(g,t)$ 进行加权汇总，得到总体平均处理效应。

在本节中，我们分别对四个场景 A、B、C、D 运行 `csdid`，并使用 `estat simple` 汇总总体 ATT。

估计中控制可观测协变量 $X_1$ 和 $X_2$，并允许使用 not-yet-treated units 作为控制组。

In [11]:
****************************************************
* Part 2.5：Callaway and Sant'Anna DID 估计
****************************************************

****************************************************
* 1. 读取示例 DGP 数据
****************************************************

use "example_dgp.dta", clear

quietly xtset id t

****************************************************
* 2. 创建 csdid 示例结果矩阵
****************************************************

capture matrix drop csdid_example_table

* 4 行对应场景 A/B/C/D
* 4 列分别为：真实 ATT、CSDID 估计值、标准误、单次样本偏误
matrix csdid_example_table = J(4, 4, .)

matrix colnames csdid_example_table = True_ATT CSDID_ATT SE Bias

matrix rownames csdid_example_table = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

****************************************************
* 3. 对四个场景分别运行 csdid
****************************************************

local row = 1

foreach sc in A B C D {

    ************************************************
    * 3.1 计算当前场景的真实 ATT
    ************************************************

    * 真实 ATT：所有已处理观测的真实 tau 平均值
    quietly summarize tau_`sc' if D_`sc' == 1
    local true_att = r(mean)

    ************************************************
    * 3.2 使用 csdid 估计当前场景
    ************************************************

    * y_`sc' 是结果变量
    * x1 x2 是可观测协变量
    * ivar(id) 指定个体变量
    * time(t) 指定时间变量
    * gvar(g_`sc') 指定首次处理时间
    * method(dripw) 使用双重稳健逆概率加权方法
    * notyet 允许使用尚未处理的个体作为控制组
    quietly csdid y_`sc' x1 x2, ///
        ivar(id) ///
        time(t) ///
        gvar(g_`sc') ///
        method(dripw) ///
        notyet

    ************************************************
    * 3.3 汇总总体 ATT
    ************************************************

    * estat simple 将 ATT(g,t) 汇总为总体 ATT
    quietly estat simple

    * r(table) 中第 1 行是估计值，第 2 行是标准误
    matrix temp = r(table)

    local cs_att = temp[1,1]
    local cs_se = temp[2,1]

    * 单次样本偏误
    local bias = `cs_att' - `true_att'

    ************************************************
    * 3.4 将结果填入矩阵
    ************************************************

    matrix csdid_example_table[`row', 1] = `true_att'
    matrix csdid_example_table[`row', 2] = `cs_att'
    matrix csdid_example_table[`row', 3] = `cs_se'
    matrix csdid_example_table[`row', 4] = `bias'

    local row = `row' + 1
}

****************************************************
* 4. 输出 csdid 示例结果
****************************************************

display "Callaway and Sant'Anna DID 示例估计结果："
matrix list csdid_example_table, format(%9.3f)










Callaway and Sant'Anna DID 示例估计结果：


csdid_example_table[4,4]
             True_ATT  CSDID_ATT         SE       Bias
Scenario_A      2.000      1.757      0.083     -0.243
Scenario_B      2.000      1.766      0.202     -0.234
Scenario_C      2.000      3.710      0.109      1.710
Scenario_D      1.368      1.414      0.076      0.046


### 2.6 Sun and Abraham 事件研究估计

本节使用 Sun and Abraham 事件研究估计方法。

在交错处理设定下，传统 TWFE event study 可能会因为错误比较 already-treated units 和 not-yet-treated units 而产生偏误。Sun and Abraham 方法通过构造 cohort-specific 的事件时间交互项，避免这类错误比较。

事件时间定义为：

$$
rel\_time_{it}=t-G_i
$$

其中：

- $rel\_time<0$ 表示处理前时期；
- $rel\_time=0$ 表示处理发生当期；
- $rel\_time>0$ 表示处理后时期。

本节将 $rel\_time=-1$ 作为基准期，因此不生成 lead1 虚拟变量。

我们分别对四个场景 A、B、C、D 运行 Sun and Abraham 事件研究估计，并整理出处理前和处理后的动态效应表。

处理前 lead 系数主要用于检查平行趋势；处理后 lag 系数用于观察处理效应的动态变化。

In [12]:
****************************************************
* Part 2.6：Sun and Abraham 事件研究估计
****************************************************

****************************************************
* 1. 读取示例 DGP 数据
****************************************************

use "example_dgp.dta", clear

quietly xtset id t

****************************************************
* 2. 删除可能已经存在的旧变量和旧矩阵
****************************************************

capture drop cohort_SA_*
capture drop never_SA_*
capture drop SA_lead*
capture drop SA_lag*

capture matrix drop SA_A SA_B SA_C SA_D SA_event_table

****************************************************
* 3. 为四个场景生成 Sun-Abraham 所需变量
****************************************************

foreach sc in A B C D {

    ************************************************
    * 3.1 生成首次处理时间变量
    ************************************************

    * cohort_SA 表示首次处理时间
    * 对 never-treated 个体，将首次处理时间设为缺失
    gen cohort_SA_`sc' = g_`sc'
    replace cohort_SA_`sc' = . if g_`sc' == 0

    ************************************************
    * 3.2 生成 never-treated 控制组变量
    ************************************************

    * never_SA=1 表示该个体从未接受处理
    gen never_SA_`sc' = (g_`sc' == 0)

    ************************************************
    * 3.3 生成处理前事件时间虚拟变量
    ************************************************

    * 注意：rel_time = -1 作为基准期，所以不生成 lead1
    * lead7 表示处理前 7 期，即 rel_time = -7
    * lead2 表示处理前 2 期，即 rel_time = -2
    gen SA_lead7_`sc' = (rel_`sc' == -7)
    gen SA_lead6_`sc' = (rel_`sc' == -6)
    gen SA_lead5_`sc' = (rel_`sc' == -5)
    gen SA_lead4_`sc' = (rel_`sc' == -4)
    gen SA_lead3_`sc' = (rel_`sc' == -3)
    gen SA_lead2_`sc' = (rel_`sc' == -2)

    ************************************************
    * 3.4 生成处理后事件时间虚拟变量
    ************************************************

    * lag0 表示处理当期，即 rel_time = 0
    * lag1 表示处理后 1 期
    * 以此类推
    gen SA_lag0_`sc' = (rel_`sc' == 0)
    gen SA_lag1_`sc' = (rel_`sc' == 1)
    gen SA_lag2_`sc' = (rel_`sc' == 2)
    gen SA_lag3_`sc' = (rel_`sc' == 3)
    gen SA_lag4_`sc' = (rel_`sc' == 4)
    gen SA_lag5_`sc' = (rel_`sc' == 5)
    gen SA_lag6_`sc' = (rel_`sc' == 6)
}

****************************************************
* 4. 运行 Sun and Abraham 事件研究估计
****************************************************

****************************************************
* 场景 A
****************************************************

quietly eventstudyinteract y_A ///
    SA_lead7_A SA_lead6_A SA_lead5_A SA_lead4_A SA_lead3_A SA_lead2_A ///
    SA_lag0_A SA_lag1_A SA_lag2_A SA_lag3_A SA_lag4_A SA_lag5_A SA_lag6_A, ///
    absorb(id t) ///
    cohort(cohort_SA_A) ///
    control_cohort(never_SA_A) ///
    vce(cluster id)

matrix SA_A = e(b_iw)

****************************************************
* 场景 B
****************************************************

quietly eventstudyinteract y_B ///
    SA_lead7_B SA_lead6_B SA_lead5_B SA_lead4_B SA_lead3_B SA_lead2_B ///
    SA_lag0_B SA_lag1_B SA_lag2_B SA_lag3_B SA_lag4_B SA_lag5_B SA_lag6_B, ///
    absorb(id t) ///
    cohort(cohort_SA_B) ///
    control_cohort(never_SA_B) ///
    vce(cluster id)

matrix SA_B = e(b_iw)

****************************************************
* 场景 C
****************************************************

quietly eventstudyinteract y_C ///
    SA_lead7_C SA_lead6_C SA_lead5_C SA_lead4_C SA_lead3_C SA_lead2_C ///
    SA_lag0_C SA_lag1_C SA_lag2_C SA_lag3_C SA_lag4_C SA_lag5_C SA_lag6_C, ///
    absorb(id t) ///
    cohort(cohort_SA_C) ///
    control_cohort(never_SA_C) ///
    vce(cluster id)

matrix SA_C = e(b_iw)

****************************************************
* 场景 D
****************************************************

quietly eventstudyinteract y_D ///
    SA_lead7_D SA_lead6_D SA_lead5_D SA_lead4_D SA_lead3_D SA_lead2_D ///
    SA_lag0_D SA_lag1_D SA_lag2_D SA_lag3_D SA_lag4_D SA_lag5_D SA_lag6_D, ///
    absorb(id t) ///
    cohort(cohort_SA_D) ///
    control_cohort(never_SA_D) ///
    vce(cluster id)

matrix SA_D = e(b_iw)

****************************************************
* 5. 整理四个场景的动态效应结果
****************************************************

* 四个场景的结果都是 1×13 矩阵
* 这里先上下合并成 4×13，再转置成 13×4
* 行表示事件时间，列表示场景
matrix SA_event_table = (SA_A \ SA_B \ SA_C \ SA_D)'

* 设置列名
matrix colnames SA_event_table = Scenario_A Scenario_B Scenario_C Scenario_D

* 设置行名
matrix rownames SA_event_table = ///
    lead7 lead6 lead5 lead4 lead3 lead2 ///
    lag0 lag1 lag2 lag3 lag4 lag5 lag6

****************************************************
* 6. 输出整理后的事件研究结果表
****************************************************

display "Sun and Abraham 事件研究示例结果：动态效应表"
matrix list SA_event_table, format(%9.3f)









(1,250 real changes made, 1,250 to missing)
(1,250 real changes made, 1,250 to missing)
(1,250 real changes made, 1,250 to missing)
(1,250 real changes made, 1,250 to missing)












Sun and Abraham 事件研究示例结果：动态效应表


SA_event_table[13,4]
       Scenario_A  Scenario_B  Scenario_C  Scenario_D
lead7      -0.002      -1.024      -1.532       0.392
lead6      -0.162      -0.312      -1.148       0.425
lead5      -0.034      -0.505      -1.193       0.068
lead4      -0.120      -0.418      -0.964       0.092
lead3      -0.094      -0.432      -0.744       0.063
lead2      -0.135      -0.315      -0.406       0.030
 lag0       1.892       1.953       2.539       2.216
 lag1       1.809       2.248       2.961       1.700
 lag2       1.829       2.479       3.407       1.330
 lag3       1.731       2.664       4.318       1.201
 lag4       1.581       2.876       4.900       0.734
 lag5       1.801       3.085       6.174       0.873
 lag6       1.495       3.357       6.692       

## Part 3. Monte Carlo Comparison of Estimators

在 Part 2 中，我们已经展示了不同估计方法在一份示例 DGP 数据上的实现方式。

本部分进一步进行正式的 Monte Carlo 比较。  
Monte Carlo 比较的核心思想是：重复生成 DGP 数据，并在每一份数据上运行不同估计方法，最后汇总各方法的统计表现。

本文正式设定：

$$
R = 1000
$$

即重复模拟 1000 次。

### 3.1 Monte Carlo 评价指标

根据作业要求，本文主要报告以下几类指标。

#### 1. 估计准确性

估计准确性主要通过以下指标衡量：

| 指标 | 含义 |
|---|---|
| Bias | 平均估计偏误 |
| RMSE | 均方根误差 |
| Median Absolute Error | 中位数绝对误差 |

三个指标的定义如下：

$$
Bias = \mathbb{E}[\hat{\theta}-\theta]
$$

$$
RMSE = \sqrt{\mathbb{E}[(\hat{\theta}-\theta)^2]}
$$

$$
Median\ Absolute\ Error = \mathrm{Median}(|\hat{\theta}-\theta|)
$$

#### 2. 推断有效性

推断有效性主要通过以下指标衡量：

| 指标 | 定义 | 含义 |
|---|---|---|
| Coverage | 95% 置信区间覆盖真实 ATT 的比例 | 越接近 0.95 越好 |
| Mean SE | Monte Carlo 中平均估计标准误 | 方法自身报告的平均不确定性 |
| MC SD | Monte Carlo 中估计值的标准差 | 估计量在重复模拟中的真实波动 |
| SE / MC SD | 平均标准误与 Monte Carlo 标准差之比 | 越接近 1 越好 |

#### 3. 动态效应估计

动态效应通过 Sun and Abraham event-study plot 展示。  
图中横轴为事件时间，纵轴为对应的事件研究系数。

重点观察：

- 处理前 lead 系数是否接近 0；
- 处理后 lag 系数是否符合 DGP 中设定的动态效应路径。

#### 4. 异质性分析

异质性分析包括：

1. 不同 cohort 的 ATT 估计；
2. 不同协变量子群的 ATT 估计，例如按照 $X_2=0$ 和 $X_2=1$ 分组。

In [13]:
****************************************************
* Part 3.1：Monte Carlo 结果表结构
****************************************************

****************************************************
* 1. 创建 Monte Carlo 主结果表的空矩阵
****************************************************

* 主表包含：
* 4 个场景 × 4 个主估计方法 = 16 行
* 8 列分别为：
* True_ATT, Mean_Estimate, Bias, RMSE, MedAE, Mean_SE, MC_SD, Coverage

capture matrix drop mc_main_table_template

matrix mc_main_table_template = J(16, 8, .)

matrix colnames mc_main_table_template = ///
    True_ATT Mean_Estimate Bias RMSE MedAE Mean_SE MC_SD Coverage

matrix rownames mc_main_table_template = ///
    A_TWFE A_PSM_DID A_Rolling_PSM A_CSDID ///
    B_TWFE B_PSM_DID B_Rolling_PSM B_CSDID ///
    C_TWFE C_PSM_DID C_Rolling_PSM C_CSDID ///
    D_TWFE D_PSM_DID D_Rolling_PSM D_CSDID

display "Monte Carlo 主结果表结构："
matrix list mc_main_table_template, format(%9.3f)

****************************************************
* 2. 创建 SCM 估计准确性表的空矩阵
****************************************************

* SCM 不直接报告标准误和覆盖率
* 因此单独报告 True_ATT, Mean_SCM, Bias, RMSE, MedAE

capture matrix drop mc_scm_table_template

matrix mc_scm_table_template = J(4, 5, .)

matrix colnames mc_scm_table_template = ///
    True_ATT Mean_SCM Bias RMSE MedAE

matrix rownames mc_scm_table_template = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

display "SCM Monte Carlo 结果表结构："
matrix list mc_scm_table_template, format(%9.3f)

****************************************************
* 3. 创建异质性分析表的空矩阵
****************************************************

* 后续用于报告不同 cohort 的 ATT
* 4 个场景 × 3 个 cohort = 12 行

capture matrix drop heterogeneity_cohort_template

matrix heterogeneity_cohort_template = J(12, 4, .)

matrix colnames heterogeneity_cohort_template = ///
    True_ATT Estimate Bias N_Treated

matrix rownames heterogeneity_cohort_template = ///
    A_g4 A_g6 A_g8 ///
    B_g4 B_g6 B_g8 ///
    C_g4 C_g6 C_g8 ///
    D_g4 D_g6 D_g8

display "cohort 异质性分析表结构："
matrix list heterogeneity_cohort_template, format(%9.3f)






Monte Carlo 主结果表结构：


mc_main_table_template[16,8]
                  True_ATT  Mean_Estim~e          Bias          RMSE
      A_TWFE             .             .             .             .
   A_PSM_DID             .             .             .             .
A_Rolling_~M             .             .             .             .
     A_CSDID             .             .             .             .
      B_TWFE             .             .             .             .
   B_PSM_DID             .             .             .             .
B_Rolling_~M             .             .             .             .
     B_CSDID             .             .             .             .
      C_TWFE             .             .             .             .
   C_PSM_DID             .             .             .             .
C_Rolling_~M             .             .             .             .
     C_CSDID             .             .             .             .
      D_TWFE             .             .       

### 3.2 Monte Carlo 主比较方法

在 Monte Carlo 主表中，本文比较以下几类标量 ATT 估计方法：

| 方法 | 是否进入 Monte Carlo 主表 | 说明 |
|---|---|---|
| TWFE | 是 | 传统基准方法 |
| 同期 PSM-DID | 是 | 匹配 DID 方法之一 |
| 滚动 PSM-DID | 是 | 匹配 DID 方法之一 |
| Callaway and Sant'Anna DID | 是 | 现代 DID 方法 |
| SCM | 部分进入 | 用于 Bias、RMSE、Median Absolute Error；标准误和覆盖率不作为主指标 |
| Sun and Abraham | 不作为主标量 ATT 主表 | 主要用于动态效应图和事件研究系数表 |

这样安排的原因是：

- TWFE、PSM-DID、滚动 PSM-DID 和 CSDID 都自然输出一个标量 ATT 和标准误，因此适合用于 Bias、RMSE、Coverage 等 Monte Carlo 指标；
- SCM 的标准 `synth` 命令不直接输出常规解析标准误，因此更适合报告估计准确性指标，而不强行报告 coverage；
- Sun and Abraham 方法主要输出动态事件时间系数，因此更适合用于 event-study plot，而不是直接压缩成单一 ATT。

### 3.3 Unified Monte Carlo Comparison

本节构建统一的 Monte Carlo 比较框架。

在每一次模拟中，我们执行以下步骤：

1. 调用 `generate_dgp`，生成一份新的模拟数据；
2. 在同一份数据上分别运行：
   - TWFE；
   - 同期 PSM-DID；
   - 滚动 PSM-DID；
   - Callaway and Sant'Anna DID；
3. 保存每种方法的估计值、标准误和真实 ATT；
4. 重复 $R=1000$ 次后，计算 Monte Carlo 评价指标。

需要注意的是，不同方法的目标参数略有不同：

- TWFE、滚动 PSM-DID 和 CSDID 对应所有处理后时期上的平均 ATT；
- 同期 PSM-DID 只比较处理前一期和处理当期，因此对应“处理刚发生时”的 ATT。

因此，在场景 D 中，同期 PSM-DID 的真实 ATT 会不同于其他方法，因为场景 D 的处理效应是动态衰减的。

In [14]:
****************************************************
* Part 3.3：统一 Monte Carlo 主程序
****************************************************

****************************************************
* 1. 定义单次 Monte Carlo 模拟程序
****************************************************

capture program drop mc_main_once

program define mc_main_once, rclass

    ************************************************
    * 1.1 生成一份新的 DGP 数据
    ************************************************

    quietly generate_dgp

    * 保存完整数据，后续各方法都从这份数据重新读取
    tempfile base
    save `base', replace

    ************************************************
    * 1.2 TWFE 估计
    ************************************************

    foreach sc in A B C D {

        use `base', clear
        quietly xtset id t

        * 真实 ATT：所有已处理观测的真实 tau 平均值
        quietly summarize tau_`sc' if D_`sc' == 1
        local true_`sc'_twfe = r(mean)

        * TWFE 回归
        quietly xtreg y_`sc' D_`sc' i.t, fe vce(cluster id)

        * 提取估计值和标准误
        local beta_`sc'_twfe = _b[D_`sc']
        local se_`sc'_twfe = _se[D_`sc']
    }

    ************************************************
    * 1.3 同期 PSM-DID 估计
    ************************************************

    foreach sc in A B C D {

        * 初始化当前场景的加权汇总变量
        local sum_n = 0
        local sum_true = 0
        local sum_est = 0
        local sum_se2 = 0

        foreach cohort in 4 6 8 {

            use `base', clear

            * 当前 cohort 的处理前一期
            local pre = `cohort' - 1

            * 只保留处理前一期和处理当期
            keep if inlist(t, `pre', `cohort')

            * 保留当前 cohort 的处理组和 clean controls
            keep if g_`sc' == `cohort' | g_`sc' == 0 | g_`sc' > `cohort'

            * 定义处理组
            gen treat = (g_`sc' == `cohort')

            * 构造处理前后变化量
            gen y_pre_tmp = y_`sc' if t == `pre'
            gen y_post_tmp = y_`sc' if t == `cohort'

            bysort id: egen y_pre = max(y_pre_tmp)
            bysort id: egen y_post = max(y_post_tmp)

            gen dy = y_post - y_pre

            keep if t == `cohort'
            drop if missing(dy)

            * 当前 cohort 的真实同期 ATT
            quietly summarize tau_`sc' if treat == 1
            local true_g = r(mean)

            * 当前 cohort 的处理组样本量
            quietly count if treat == 1
            local n_g = r(N)

            * 倾向得分匹配 DID
            * capture 的作用是避免极少数模拟中因为 overlap 问题中断整个 Monte Carlo
            capture quietly teffects psmatch (dy) (treat x1 x2), atet

            if _rc == 0 {

                matrix b = e(b)
                matrix V = e(V)

                local att_g = b[1,1]
                local se_g = sqrt(V[1,1])

                * 按处理组样本量加权汇总
                local sum_n = `sum_n' + `n_g'
                local sum_true = `sum_true' + `n_g' * `true_g'
                local sum_est = `sum_est' + `n_g' * `att_g'
                local sum_se2 = `sum_se2' + (`n_g'^2) * (`se_g'^2)
            }
        }

        * 当前场景的同期 PSM-DID 汇总结果
        if `sum_n' > 0 {
            local true_`sc'_psm = `sum_true' / `sum_n'
            local beta_`sc'_psm = `sum_est' / `sum_n'
            local se_`sc'_psm = sqrt(`sum_se2') / `sum_n'
        }
        else {
            local true_`sc'_psm = .
            local beta_`sc'_psm = .
            local se_`sc'_psm = .
        }
    }

    ************************************************
    * 1.4 滚动 PSM-DID 估计
    ************************************************

    foreach sc in A B C D {

        * 初始化当前场景的加权汇总变量
        local sum_n = 0
        local sum_true = 0
        local sum_est = 0
        local sum_se2 = 0

        foreach cohort in 4 6 8 {

            * 当前 cohort 的处理前一期
            local pre = `cohort' - 1

            * 对当前 cohort 的所有处理后时期循环
            forvalues post = `cohort'/$T {

                use `base', clear

                * 只保留处理前一期和当前 post 期
                keep if t == `pre' | t == `post'

                * 保留当前 cohort 的处理组和当前 post 期仍未处理的 clean controls
                keep if g_`sc' == `cohort' | g_`sc' == 0 | g_`sc' > `post'

                * 定义处理组
                gen treat = (g_`sc' == `cohort')

                * 构造从处理前一期到当前 post 期的变化量
                gen y_pre_tmp = y_`sc' if t == `pre'
                gen y_post_tmp = y_`sc' if t == `post'

                bysort id: egen y_pre = max(y_pre_tmp)
                bysort id: egen y_post = max(y_post_tmp)

                gen dy = y_post - y_pre

                keep if t == `post'
                drop if missing(dy)

                * 当前 (cohort, post) 单元的真实 ATT
                quietly summarize tau_`sc' if treat == 1
                local true_gt = r(mean)

                * 当前单元处理组样本量
                quietly count if treat == 1
                local n_gt = r(N)

                * 倾向得分匹配 DID
                * capture 用于避免极少数单元 common support 不足导致模拟中断
                capture quietly teffects psmatch (dy) (treat x1 x2), atet pstolerance(1e-10)

                if _rc == 0 {

                    matrix b = e(b)
                    matrix V = e(V)

                    local att_gt = b[1,1]
                    local se_gt = sqrt(V[1,1])

                    * 按处理组样本量加权汇总
                    local sum_n = `sum_n' + `n_gt'
                    local sum_true = `sum_true' + `n_gt' * `true_gt'
                    local sum_est = `sum_est' + `n_gt' * `att_gt'
                    local sum_se2 = `sum_se2' + (`n_gt'^2) * (`se_gt'^2)
                }
            }
        }

        * 当前场景的滚动 PSM-DID 汇总结果
        if `sum_n' > 0 {
            local true_`sc'_rpsm = `sum_true' / `sum_n'
            local beta_`sc'_rpsm = `sum_est' / `sum_n'
            local se_`sc'_rpsm = sqrt(`sum_se2') / `sum_n'
        }
        else {
            local true_`sc'_rpsm = .
            local beta_`sc'_rpsm = .
            local se_`sc'_rpsm = .
        }
    }

    ************************************************
    * 1.5 CSDID 估计
    ************************************************

    foreach sc in A B C D {

        use `base', clear
        quietly xtset id t

        * 真实 ATT：所有已处理观测的真实 tau 平均值
        quietly summarize tau_`sc' if D_`sc' == 1
        local true_`sc'_csdid = r(mean)

        * CSDID 估计
        capture quietly csdid y_`sc' x1 x2, ///
            ivar(id) ///
            time(t) ///
            gvar(g_`sc') ///
            method(dripw) ///
            notyet

        if _rc == 0 {

            quietly estat simple

            matrix temp = r(table)

            local beta_`sc'_csdid = temp[1,1]
            local se_`sc'_csdid = temp[2,1]
        }
        else {
            local beta_`sc'_csdid = .
            local se_`sc'_csdid = .
        }
    }

    ************************************************
    * 1.6 返回所有方法的结果
    ************************************************

    foreach sc in A B C D {

        ************************************************
        * TWFE
        ************************************************
        return scalar true_`sc'_twfe = `true_`sc'_twfe'
        return scalar beta_`sc'_twfe = `beta_`sc'_twfe'
        return scalar se_`sc'_twfe = `se_`sc'_twfe'

        ************************************************
        * 同期 PSM-DID
        ************************************************
        return scalar true_`sc'_psm = `true_`sc'_psm'
        return scalar beta_`sc'_psm = `beta_`sc'_psm'
        return scalar se_`sc'_psm = `se_`sc'_psm'

        ************************************************
        * 滚动 PSM-DID
        ************************************************
        return scalar true_`sc'_rpsm = `true_`sc'_rpsm'
        return scalar beta_`sc'_rpsm = `beta_`sc'_rpsm'
        return scalar se_`sc'_rpsm = `se_`sc'_rpsm'

        ************************************************
        * CSDID
        ************************************************
        return scalar true_`sc'_csdid = `true_`sc'_csdid'
        return scalar beta_`sc'_csdid = `beta_`sc'_csdid'
        return scalar se_`sc'_csdid = `se_`sc'_csdid'
    }

end

### 3.4 运行统一 Monte Carlo 并汇总结果

在上一节中，我们已经定义了单次模拟程序 `mc_main_once`。

本节正式运行 Monte Carlo 模拟。每一次模拟都会：

1. 生成一份新的 DGP 数据；
2. 在同一份数据上运行 TWFE、同期 PSM-DID、滚动 PSM-DID 和 CSDID；
3. 保存每种方法的估计值、标准误和真实 ATT。

重复模拟 $R=1000$ 次后，本文计算以下指标：

$$
Bias = \mathbb{E}[\hat{\theta}-\theta]
$$

$$
RMSE = \sqrt{\mathbb{E}[(\hat{\theta}-\theta)^2]}
$$

$$
Median\ Absolute\ Error = \mathrm{Median}(|\hat{\theta}-\theta|)
$$

同时，为了评估推断有效性，本文还计算：

$$
Coverage = Pr(\theta \in [\hat{\theta}-1.96SE,\hat{\theta}+1.96SE])
$$

并比较平均标准误与 Monte Carlo 标准差：

$$
SE/MCSD = \frac{\overline{SE}}{SD(\hat{\theta})}
$$

In [15]:
****************************************************
* Part 3.4：运行统一 Monte Carlo 模拟
****************************************************

simulate ///
    true_A_twfe = r(true_A_twfe) beta_A_twfe = r(beta_A_twfe) se_A_twfe = r(se_A_twfe) ///
    true_A_psm = r(true_A_psm) beta_A_psm = r(beta_A_psm) se_A_psm = r(se_A_psm) ///
    true_A_rpsm = r(true_A_rpsm) beta_A_rpsm = r(beta_A_rpsm) se_A_rpsm = r(se_A_rpsm) ///
    true_A_csdid = r(true_A_csdid) beta_A_csdid = r(beta_A_csdid) se_A_csdid = r(se_A_csdid) ///
    true_B_twfe = r(true_B_twfe) beta_B_twfe = r(beta_B_twfe) se_B_twfe = r(se_B_twfe) ///
    true_B_psm = r(true_B_psm) beta_B_psm = r(beta_B_psm) se_B_psm = r(se_B_psm) ///
    true_B_rpsm = r(true_B_rpsm) beta_B_rpsm = r(beta_B_rpsm) se_B_rpsm = r(se_B_rpsm) ///
    true_B_csdid = r(true_B_csdid) beta_B_csdid = r(beta_B_csdid) se_B_csdid = r(se_B_csdid) ///
    true_C_twfe = r(true_C_twfe) beta_C_twfe = r(beta_C_twfe) se_C_twfe = r(se_C_twfe) ///
    true_C_psm = r(true_C_psm) beta_C_psm = r(beta_C_psm) se_C_psm = r(se_C_psm) ///
    true_C_rpsm = r(true_C_rpsm) beta_C_rpsm = r(beta_C_rpsm) se_C_rpsm = r(se_C_rpsm) ///
    true_C_csdid = r(true_C_csdid) beta_C_csdid = r(beta_C_csdid) se_C_csdid = r(se_C_csdid) ///
    true_D_twfe = r(true_D_twfe) beta_D_twfe = r(beta_D_twfe) se_D_twfe = r(se_D_twfe) ///
    true_D_psm = r(true_D_psm) beta_D_psm = r(beta_D_psm) se_D_psm = r(se_D_psm) ///
    true_D_rpsm = r(true_D_rpsm) beta_D_rpsm = r(beta_D_rpsm) se_D_rpsm = r(se_D_rpsm) ///
    true_D_csdid = r(true_D_csdid) beta_D_csdid = r(beta_D_csdid) se_D_csdid = r(se_D_csdid), ///
    reps($R) seed(20260529) dots(50): mc_main_once

****************************************************
* 保存原始 Monte Carlo 结果
****************************************************

save "mc_main_raw_results.dta", replace



        Command: mc_main_once
    true_A_twfe: r(true_A_twfe)
    beta_A_twfe: r(beta_A_twfe)
      se_A_twfe: r(se_A_twfe)
     true_A_psm: r(true_A_psm)
     beta_A_psm: r(beta_A_psm)
       se_A_psm: r(se_A_psm)
    true_A_rpsm: r(true_A_rpsm)
    beta_A_rpsm: r(beta_A_rpsm)
      se_A_rpsm: r(se_A_rpsm)
   true_A_csdid: r(true_A_csdid)
   beta_A_csdid: r(beta_A_csdid)
     se_A_csdid: r(se_A_csdid)
    true_B_twfe: r(true_B_twfe)
    beta_B_twfe: r(beta_B_twfe)
      se_B_twfe: r(se_B_twfe)
     true_B_psm: r(true_B_psm)
     beta_B_psm: r(beta_B_psm)
       se_B_psm: r(se_B_psm)
    true_B_rpsm: r(true_B_rpsm)
    beta_B_rpsm: r(beta_B_rpsm)
      se_B_rpsm: r(se_B_rpsm)
   true_B_csdid: r(true_B_csdid)
   beta_B_csdid: r(beta_B_csdid)
     se_B_csdid: r(se_B_csdid)
    true_C_twfe: r(true_C_twfe)
    beta_C_twfe: r(beta_C_twfe)
      se_C_twfe: r(se_C_twfe)
     true_C_psm: r(true_C_psm)
     beta_C_psm: r(beta_C_psm)
       se_C_psm: r(se_C_psm)
    true_C_rpsm: r(true_C_rpsm)

In [16]:
****************************************************
* Part 3.5：计算 Monte Carlo 评价指标
****************************************************

****************************************************
* 1. 生成误差、绝对误差、平方误差和覆盖率变量
****************************************************

foreach sc in A B C D {

    foreach m in twfe psm rpsm csdid {

        * 估计误差
        gen err_`sc'_`m' = beta_`sc'_`m' - true_`sc'_`m'

        * 绝对误差
        gen abserr_`sc'_`m' = abs(err_`sc'_`m')

        * 平方误差
        gen sqerr_`sc'_`m' = err_`sc'_`m'^2

        * 95% 置信区间是否覆盖真实 ATT
        gen cover_`sc'_`m' = ///
            (true_`sc'_`m' >= beta_`sc'_`m' - 1.96 * se_`sc'_`m' ///
          &  true_`sc'_`m' <= beta_`sc'_`m' + 1.96 * se_`sc'_`m')
    }
}

****************************************************
* 2. 创建 Monte Carlo 主结果矩阵
****************************************************

capture matrix drop mc_main_table

matrix mc_main_table = J(16, 8, .)

matrix colnames mc_main_table = ///
    True_ATT Mean_Estimate Bias RMSE MedAE Mean_SE MC_SD Coverage

matrix rownames mc_main_table = ///
    A_TWFE A_PSM_DID A_Rolling_PSM A_CSDID ///
    B_TWFE B_PSM_DID B_Rolling_PSM B_CSDID ///
    C_TWFE C_PSM_DID C_Rolling_PSM C_CSDID ///
    D_TWFE D_PSM_DID D_Rolling_PSM D_CSDID

****************************************************
* 3. 汇总每个场景和每种方法的结果
****************************************************

local row = 1

foreach sc in A B C D {

    foreach m in twfe psm rpsm csdid {

        ************************************************
        * 3.1 平均真实 ATT
        ************************************************

        quietly summarize true_`sc'_`m'
        local mean_true = r(mean)

        ************************************************
        * 3.2 平均估计值和 Monte Carlo 标准差
        ************************************************

        quietly summarize beta_`sc'_`m'
        local mean_beta = r(mean)
        local mc_sd = r(sd)

        ************************************************
        * 3.3 Bias
        ************************************************

        quietly summarize err_`sc'_`m'
        local bias = r(mean)

        ************************************************
        * 3.4 RMSE
        ************************************************

        quietly summarize sqerr_`sc'_`m'
        local rmse = sqrt(r(mean))

        ************************************************
        * 3.5 Median Absolute Error
        ************************************************

        quietly summarize abserr_`sc'_`m', detail
        local medae = r(p50)

        ************************************************
        * 3.6 Mean SE
        ************************************************

        quietly summarize se_`sc'_`m'
        local mean_se = r(mean)

        ************************************************
        * 3.7 Coverage
        ************************************************

        quietly summarize cover_`sc'_`m'
        local coverage = r(mean)

        ************************************************
        * 3.8 将结果填入矩阵
        ************************************************

        matrix mc_main_table[`row',1] = `mean_true'
        matrix mc_main_table[`row',2] = `mean_beta'
        matrix mc_main_table[`row',3] = `bias'
        matrix mc_main_table[`row',4] = `rmse'
        matrix mc_main_table[`row',5] = `medae'
        matrix mc_main_table[`row',6] = `mean_se'
        matrix mc_main_table[`row',7] = `mc_sd'
        matrix mc_main_table[`row',8] = `coverage'

        local row = `row' + 1
    }
}

****************************************************
* 4. 输出 Monte Carlo 主结果表
****************************************************

display "Monte Carlo 主结果表：估计准确性与推断有效性"
matrix list mc_main_table, format(%9.3f)

****************************************************
* 5. 保存包含评价指标变量的 Monte Carlo 数据
****************************************************

save "mc_main_with_metrics.dta", replace


(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)







Monte Carlo 主结果表：估计准确性与推断有效性


mc_main_table[16,8]
                  True_ATT  Mean_Estim~e          Bias          RMSE
      A_TWFE         2.000         1.999        -0.001         0.053
   A_PSM_DID         2.000         1.999        -0.001         0.125
A_Rolling_~M         2.000         2.000        -0.000         0.101
     A_CSDID         2.000         2.000        -0.000         0.083
      B_TWFE         2.000         2.392         0.392         0.396
   B_PSM_DID         2.000         2.004         0.004         0.227
B_Rolling_~M         2.000         2.088         0.088         0.313
     B_CSDID         2.000         2.008         0.00

In [17]:
****************************************************
* Part 3.6：标准误与 Monte Carlo 标准差比较
****************************************************

capture matrix drop mc_se_mcsd_table

matrix mc_se_mcsd_table = J(16, 3, .)

matrix colnames mc_se_mcsd_table = Mean_SE MC_SD SE_over_MCSD

matrix rownames mc_se_mcsd_table = ///
    A_TWFE A_PSM_DID A_Rolling_PSM A_CSDID ///
    B_TWFE B_PSM_DID B_Rolling_PSM B_CSDID ///
    C_TWFE C_PSM_DID C_Rolling_PSM C_CSDID ///
    D_TWFE D_PSM_DID D_Rolling_PSM D_CSDID

local row = 1

foreach sc in A B C D {

    foreach m in twfe psm rpsm csdid {

        quietly summarize se_`sc'_`m'
        local mean_se = r(mean)

        quietly summarize beta_`sc'_`m'
        local mc_sd = r(sd)

        local ratio = `mean_se' / `mc_sd'

        matrix mc_se_mcsd_table[`row',1] = `mean_se'
        matrix mc_se_mcsd_table[`row',2] = `mc_sd'
        matrix mc_se_mcsd_table[`row',3] = `ratio'

        local row = `row' + 1
    }
}

display "标准误 vs Monte Carlo 标准差："
matrix list mc_se_mcsd_table, format(%9.3f)








标准误 vs Monte Carlo 标准差：


mc_se_mcsd_table[16,3]
                   Mean_SE         MC_SD  SE_over_MCSD
      A_TWFE         0.051         0.053         0.966
   A_PSM_DID         0.124         0.125         0.989
A_Rolling_~M         0.058         0.101         0.576
     A_CSDID         0.081         0.083         0.973
      B_TWFE         0.057         0.055         1.042
   B_PSM_DID         0.215         0.227         0.950
B_Rolling_~M         0.174         0.301         0.577
     B_CSDID         0.157         0.260         0.602
      C_TWFE         0.088         0.072         1.224
   C_PSM_DID         0.128         0.130         0.983
C_Rolling_~M         0.068         0.126         0.537
     C_CSDID         0.109         0.106         1.027
      D_TWFE         0.058         0.051         1.137
   D_PSM_DID         0.124         0.125         0.996
D_Rolling_~M         0.058         0.102         0.568
     D_CSDID         0.081         0.080         1.010


### 3.7 Coverage Table

本节单独整理 95% 置信区间覆盖率表。

虽然 Part 3.5 的 Monte Carlo 主结果表中已经包含 `Coverage` 一列，但为了更清楚地展示推断有效性，本节将不同场景和不同估计方法的覆盖率单独整理成表格。

覆盖率定义为：

$$
Coverage = Pr\left(\theta \in [\hat{\theta}-1.96SE,\ \hat{\theta}+1.96SE]\right)
$$

其中：

- $\theta$ 表示 DGP 中的真实 ATT；
- $\hat{\theta}$ 表示估计方法得到的 ATT；
- $SE$ 表示该估计方法报告的标准误。

如果估计方法的标准误是准确的，并且估计量近似正态，那么 95% 置信区间覆盖率应当接近 0.95。

如果覆盖率明显低于 0.95，说明该方法的推断可能存在问题，例如估计偏误较大、标准误低估，或者识别假设不成立。

In [18]:
****************************************************
* Part 3.7：Coverage Table
****************************************************

****************************************************
* 1. 读取 Monte Carlo 原始结果
****************************************************

use "mc_main_raw_results.dta", clear

****************************************************
* 2. 重新生成 coverage 变量
* 说明：为了让本 cell 可以独立运行，这里重新计算 coverage
****************************************************

foreach sc in A B C D {

    foreach m in twfe psm rpsm csdid {

        * 如果之前已经存在同名 coverage 变量，先删除
        capture drop cover_`sc'_`m'

        * 95% 置信区间是否覆盖真实 ATT
        gen cover_`sc'_`m' = ///
            (true_`sc'_`m' >= beta_`sc'_`m' - 1.96 * se_`sc'_`m' ///
          &  true_`sc'_`m' <= beta_`sc'_`m' + 1.96 * se_`sc'_`m')
    }
}

****************************************************
* 3. 创建 coverage table
****************************************************

capture matrix drop coverage_table

* 4 行对应场景 A/B/C/D
* 4 列对应四种主估计方法
matrix coverage_table = J(4, 4, .)

matrix colnames coverage_table = TWFE PSM_DID Rolling_PSM CSDID

matrix rownames coverage_table = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

****************************************************
* 4. 计算每个场景、每种方法的覆盖率
****************************************************

local row = 1

foreach sc in A B C D {

    local col = 1

    foreach m in twfe psm rpsm csdid {

        quietly summarize cover_`sc'_`m'
        matrix coverage_table[`row', `col'] = r(mean)

        local col = `col' + 1
    }

    local row = `row' + 1
}

****************************************************
* 5. 输出 coverage table
****************************************************

display "95% 置信区间覆盖率表："
matrix list coverage_table, format(%9.3f)

****************************************************
* 6. 保存带 coverage 变量的结果数据
****************************************************

save "mc_coverage_results.dta", replace


(simulate: mc_main_once)








95% 置信区间覆盖率表：


coverage_table[4,4]
                   TWFE      PSM_DID  Rolling_PSM        CSDID
Scenario_A        0.944        0.951        0.753        0.937
Scenario_B        0.000        0.913        0.712        0.759
Scenario_C        0.000        0.357        0.000        0.000
Scenario_D        0.000        0.950        0.740        0.955

(file mc_coverage_results.dta not found)
file mc_coverage_results.dta saved


### Interpretation of Coverage Results

Coverage 接近 0.95 表示该方法的 95% 置信区间能够较好覆盖真实 ATT，说明推断较为可靠。

在场景 A 中，平行趋势成立且处理效应同质，因此表现良好的估计方法应当具有接近 0.95 的覆盖率。

在场景 B 中，处理分配与可观测协变量相关。如果某种方法没有充分控制或匹配这些协变量，覆盖率可能下降。

在场景 C 中，存在不可观测时变混淆，因此即使某些方法报告了较小的标准误，其置信区间也可能无法覆盖真实 ATT，覆盖率可能明显低于 0.95。

在场景 D 中，存在动态处理效应和 cohort 异质性。传统 TWFE 可能由于错误比较和动态异质效应而出现较低覆盖率，而现代 DID 方法通常应当表现更好。

### 3.8 Bias-Variance Tradeoff Plot

本节绘制偏误-方差权衡图。

在 Monte Carlo 模拟中，一个估计方法可能具有较小的偏误，但估计值波动较大；也可能估计较稳定，但存在较大系统性偏误。因此，仅报告 Bias 或 RMSE 不足以完整展示估计方法的表现。

本节使用以下两个指标绘制偏误-方差权衡图：

- 横轴：Monte Carlo 标准差，即 $SD(\hat{\theta})$；
- 纵轴：平均偏误的绝对值，即 $|Bias|$。

其中：

$$
Bias = E(\hat{\theta}-\theta)
$$

$$
MC\_SD = SD(\hat{\theta})
$$

图中每个点代表一种估计方法。  
越靠近左下角，说明该方法同时具有较小偏误和较小估计波动，表现越好。

In [19]:
****************************************************
* Part 3.8：Bias-Variance Tradeoff Plot
****************************************************

****************************************************
* 1. 读取 Monte Carlo 原始结果
****************************************************

use "mc_main_raw_results.dta", clear

****************************************************
* 2. 创建偏误-方差权衡图所需数据
****************************************************

tempfile bv_data

postfile handle ///
    str1 scenario ///
    str15 method ///
    double bias ///
    double abs_bias ///
    double mc_sd ///
    double rmse ///
    using `bv_data', replace

foreach sc in A B C D {

    foreach m in twfe psm rpsm csdid {

        ************************************************
        * 2.1 生成临时误差变量
        ************************************************

        capture drop temp_err
        capture drop temp_sqerr

        gen temp_err = beta_`sc'_`m' - true_`sc'_`m'
        gen temp_sqerr = temp_err^2

        ************************************************
        * 2.2 计算 Bias
        ************************************************

        quietly summarize temp_err
        local bias = r(mean)
        local abs_bias = abs(`bias')

        ************************************************
        * 2.3 计算 Monte Carlo 标准差
        ************************************************

        quietly summarize beta_`sc'_`m'
        local mc_sd = r(sd)

        ************************************************
        * 2.4 计算 RMSE
        ************************************************

        quietly summarize temp_sqerr
        local rmse = sqrt(r(mean))

        ************************************************
        * 2.5 设置方法名称
        ************************************************

        if "`m'" == "twfe"  local method_name "TWFE"
        if "`m'" == "psm"   local method_name "PSM-DID"
        if "`m'" == "rpsm"  local method_name "Rolling-PSM"
        if "`m'" == "csdid" local method_name "CSDID"

        ************************************************
        * 2.6 保存当前结果
        ************************************************

        post handle ("`sc'") ("`method_name'") (`bias') (`abs_bias') (`mc_sd') (`rmse')
    }
}

postclose handle

****************************************************
* 3. 读取整理后的 bias-variance 数据
****************************************************

use `bv_data', clear

label variable scenario "Scenario"
label variable method "Method"
label variable bias "Bias"
label variable abs_bias "Absolute Bias"
label variable mc_sd "Monte Carlo SD"
label variable rmse "RMSE"

****************************************************
* 4. 保存 bias-variance 数据
****************************************************

save "bias_variance_tradeoff_data.dta", replace

****************************************************
* 5. 创建图片文件夹
****************************************************

capture mkdir "figures"

****************************************************
* 6. 绘制四个场景的偏误-方差权衡图
* 说明：使用 quietly twoway 避免 ipynb 直接渲染图形
****************************************************

****************************************************
* 场景 A
****************************************************

quietly twoway ///
    (scatter abs_bias mc_sd if scenario == "A", ///
        mlabel(method) mlabposition(12)), ///
    xtitle("Monte Carlo SD") ///
    ytitle("Absolute Bias") ///
    title("Bias-Variance Tradeoff: Scenario A") ///
    legend(off)

graph export "figures/bias_variance_scenario_A.png", replace width(1600)
graph drop _all

****************************************************
* 场景 B
****************************************************

quietly twoway ///
    (scatter abs_bias mc_sd if scenario == "B", ///
        mlabel(method) mlabposition(12)), ///
    xtitle("Monte Carlo SD") ///
    ytitle("Absolute Bias") ///
    title("Bias-Variance Tradeoff: Scenario B") ///
    legend(off)

graph export "figures/bias_variance_scenario_B.png", replace width(1600)
graph drop _all

****************************************************
* 场景 C
****************************************************

quietly twoway ///
    (scatter abs_bias mc_sd if scenario == "C", ///
        mlabel(method) mlabposition(12)), ///
    xtitle("Monte Carlo SD") ///
    ytitle("Absolute Bias") ///
    title("Bias-Variance Tradeoff: Scenario C") ///
    legend(off)

graph export "figures/bias_variance_scenario_C.png", replace width(1600)
graph drop _all

****************************************************
* 场景 D
****************************************************

quietly twoway ///
    (scatter abs_bias mc_sd if scenario == "D", ///
        mlabel(method) mlabposition(12)), ///
    xtitle("Monte Carlo SD") ///
    ytitle("Absolute Bias") ///
    title("Bias-Variance Tradeoff: Scenario D") ///
    legend(off)

graph export "figures/bias_variance_scenario_D.png", replace width(1600)
graph drop _all

****************************************************
* 7. 检查图片是否成功导出
****************************************************

dir "figures/bias_variance_*.png"

display "偏误-方差权衡图已经导出到 figures 文件夹。"


(simulate: mc_main_once)


(file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000007.tmp not found)

(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)









(file bias_variance_tradeoff_data.dta not found)
file bias_variance_tradeoff_data.dta saved



(file figures/bias_variance_scenario_A.png not found)
file figures/bias_variance_scenario_A.png saved as PNG format



(file figures/bias_variance_scenario_B.png not found)
file figures/bias_variance_scenario_B.png saved as PNG format



(file figures/bias_variance_scenario_C.png not found)
file figures/bias_variance_scenario_C.png saved as PNG format



(file figures/bias_variance_scenario_D.png not found)
file figures/bias_variance_scenario_D.png saved as PNG format


  76.4k   6/01/26 15:37  bias_variance_scenario_A.png
  76.1k   6/01/26 15:37  bias_variance_sc

### Bias-Variance Tradeoff: Scenario A

<img src="./figures/bias_variance_scenario_A.png" width="750">

### Bias-Variance Tradeoff: Scenario B

<img src="./figures/bias_variance_scenario_B.png" width="750">

### Bias-Variance Tradeoff: Scenario C

<img src="./figures/bias_variance_scenario_C.png" width="750">

### Bias-Variance Tradeoff: Scenario D

<img src="./figures/bias_variance_scenario_D.png" width="750">

### 偏误-方差权衡图解释

偏误-方差权衡图用于比较不同估计方法在 Monte Carlo 模拟中的系统性偏误和估计波动。

图中的横轴是 Monte Carlo 标准差，即估计值在重复模拟中的标准差，用来衡量估计量的波动程度。横轴越小，说明估计结果越稳定。

图中的纵轴是绝对偏误，即平均估计值与真实 ATT 之间差异的绝对值。纵轴越小，说明估计结果越接近真实处理效应。

因此，越靠近左下角的方法，说明它同时具有较小偏误和较小波动，整体表现越好。

在场景 A 中，处理分配是随机的，平行趋势成立，且处理效应同质。因此，大多数方法理论上都应该表现较好，偏误和波动都应相对较小。

在场景 B 中，处理时间与可观测协变量相关。如果方法能够有效控制或匹配这些协变量，那么它的偏误应该比 TWFE 更小。因此，匹配类方法和 CSDID 理论上应当比 TWFE 更靠近左下角。

在场景 C 中，处理时间与不可观测时变混淆因素相关。由于该不可观测因素同时影响处理分配和未处理结果趋势，各类方法都可能出现较大偏误。因此，场景 C 中的点可能整体更靠上，说明识别假设受到破坏。

在场景 D 中，处理效应具有动态衰减和 cohort 异质性。传统 TWFE 可能因为错误比较和异质处理效应而产生偏误；相比之下，CSDID 等现代 DID 方法更适合处理交错处理和异质效应，因此理论上应该表现更好。

## Part 4. Dynamic Effects and Event-study Plot

在 Part 3 中，我们已经使用 Monte Carlo 模拟比较了不同估计方法的总体 ATT 表现。

本部分进一步考察动态处理效应，也就是处理前后不同相对时期的估计系数。

我们使用 Sun and Abraham 事件研究估计方法，构造如下事件时间：

$$
rel\_time_{it}=t-G_i
$$

其中：

- $rel\_time<0$ 表示处理前；
- $rel\_time=0$ 表示处理发生当期；
- $rel\_time>0$ 表示处理后。

本部分将 $rel\_time=-1$ 作为基准期，因此事件研究图中不显示 $-1$。

事件研究图主要用于检查两点：

1. 处理前 lead 系数是否接近 0，用于判断平行趋势是否合理；
2. 处理后 lag 系数是否符合 DGP 中设定的动态处理效应路径。

特别是在场景 D 中，DGP 设定了动态衰减的处理效应，因此事件研究图应当显示处理效应随事件时间逐渐下降。

In [20]:
****************************************************
* Part 4.1：生成 Sun and Abraham 事件研究系数
****************************************************

****************************************************
* 1. 读取示例 DGP 数据
****************************************************

use "example_dgp.dta", clear

quietly xtset id t

****************************************************
* 2. 删除可能已经存在的旧变量和旧矩阵
****************************************************

capture drop cohort_SA_*
capture drop never_SA_*
capture drop SA_lead*
capture drop SA_lag*

capture matrix drop SA_A SA_B SA_C SA_D
capture matrix drop V_A V_B V_C V_D

****************************************************
* 3. 为四个场景生成 Sun-Abraham 所需变量
****************************************************

foreach sc in A B C D {

    ************************************************
    * 3.1 生成首次处理时间变量
    ************************************************

    * cohort_SA 表示首次处理时间
    * never-treated 个体的首次处理时间设为缺失
    gen cohort_SA_`sc' = g_`sc'
    replace cohort_SA_`sc' = . if g_`sc' == 0

    ************************************************
    * 3.2 生成 never-treated 控制组变量
    ************************************************

    * never_SA=1 表示该个体从未接受处理
    gen never_SA_`sc' = (g_`sc' == 0)

    ************************************************
    * 3.3 生成处理前事件时间虚拟变量
    ************************************************

    * rel_time = -1 作为基准期，所以不生成 lead1
    gen SA_lead7_`sc' = (rel_`sc' == -7)
    gen SA_lead6_`sc' = (rel_`sc' == -6)
    gen SA_lead5_`sc' = (rel_`sc' == -5)
    gen SA_lead4_`sc' = (rel_`sc' == -4)
    gen SA_lead3_`sc' = (rel_`sc' == -3)
    gen SA_lead2_`sc' = (rel_`sc' == -2)

    ************************************************
    * 3.4 生成处理后事件时间虚拟变量
    ************************************************

    gen SA_lag0_`sc' = (rel_`sc' == 0)
    gen SA_lag1_`sc' = (rel_`sc' == 1)
    gen SA_lag2_`sc' = (rel_`sc' == 2)
    gen SA_lag3_`sc' = (rel_`sc' == 3)
    gen SA_lag4_`sc' = (rel_`sc' == 4)
    gen SA_lag5_`sc' = (rel_`sc' == 5)
    gen SA_lag6_`sc' = (rel_`sc' == 6)
}

****************************************************
* 4. 运行 Sun and Abraham 事件研究估计
****************************************************

foreach sc in A B C D {

    quietly eventstudyinteract y_`sc' ///
        SA_lead7_`sc' SA_lead6_`sc' SA_lead5_`sc' SA_lead4_`sc' SA_lead3_`sc' SA_lead2_`sc' ///
        SA_lag0_`sc' SA_lag1_`sc' SA_lag2_`sc' SA_lag3_`sc' SA_lag4_`sc' SA_lag5_`sc' SA_lag6_`sc', ///
        absorb(id t) ///
        cohort(cohort_SA_`sc') ///
        control_cohort(never_SA_`sc') ///
        vce(cluster id)

    * 保存事件研究系数
    matrix SA_`sc' = e(b_iw)

    * 保存方差协方差矩阵，用于计算标准误
    matrix V_`sc' = e(V_iw)
}

****************************************************
* 5. 将矩阵结果整理成长数据，方便画图
****************************************************

tempfile event_results

postfile handle str1 scenario int event_time double coef se using `event_results', replace

foreach sc in A B C D {

    matrix B = SA_`sc'
    matrix V = V_`sc'

    * 事件时间顺序：
    * -7, -6, -5, -4, -3, -2, 0, 1, 2, 3, 4, 5, 6
    local j = 1

    foreach ev in -7 -6 -5 -4 -3 -2 0 1 2 3 4 5 6 {

        local b = B[1, `j']
        local s = sqrt(V[`j', `j'])

        post handle ("`sc'") (`ev') (`b') (`s')

        local j = `j' + 1
    }
}

postclose handle

use `event_results', clear

****************************************************
* 6. 生成置信区间
****************************************************

gen ci_low = coef - 1.96 * se
gen ci_high = coef + 1.96 * se

label variable event_time "Event Time"
label variable coef "Estimated Coefficient"
label variable ci_low "95% CI Lower Bound"
label variable ci_high "95% CI Upper Bound"

****************************************************
* 7. 保存事件研究系数数据
****************************************************

save "event_study_coefficients.dta", replace

****************************************************
* 8. 输出事件研究系数表
****************************************************

display "Sun and Abraham 事件研究系数数据已生成："
tab scenario










(1,250 real changes made, 1,250 to missing)
(1,250 real changes made, 1,250 to missing)
(1,250 real changes made, 1,250 to missing)
(1,250 real changes made, 1,250 to missing)



(file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000008.tmp not found)










(file event_study_coefficients.dta not found)
file event_study_coefficients.dta saved

Sun and Abraham 事件研究系数数据已生成：


   scenario |      Freq.     Percent        Cum.
------------+-----------------------------------
          A |         13       25.00       25.00
          B |         13       25.00       50.00
          C |         13       25.00       75.00
          D |         13       25.00      100.00
------------+-----------------------------------
      Total |         52      100.00


In [21]:
****************************************************
* Part 4.2：绘制事件研究图
****************************************************

****************************************************
* 1. 读取事件研究系数数据
****************************************************

use "event_study_coefficients.dta", clear

****************************************************
* 2. 创建 figures 文件夹
****************************************************

capture mkdir "figures"

****************************************************
* 3. 场景 A：基准情形
****************************************************

quietly twoway ///
    (rcap ci_low ci_high event_time if scenario == "A") ///
    (connected coef event_time if scenario == "A"), ///
    yline(0) ///
    xline(-1) ///
    xtitle("Event Time") ///
    ytitle("Estimated Effect") ///
    title("Event-study Plot: Scenario A") ///
    legend(off)

graph export "figures/event_study_scenario_A.png", replace
graph drop _all

****************************************************
* 4. 场景 B：可观测选择偏误
****************************************************

quietly twoway ///
    (rcap ci_low ci_high event_time if scenario == "B") ///
    (connected coef event_time if scenario == "B"), ///
    yline(0) ///
    xline(-1) ///
    xtitle("Event Time") ///
    ytitle("Estimated Effect") ///
    title("Event-study Plot: Scenario B") ///
    legend(off)

graph export "figures/event_study_scenario_B.png", replace
graph drop _all

****************************************************
* 5. 场景 C：不可观测时变混淆
****************************************************

quietly twoway ///
    (rcap ci_low ci_high event_time if scenario == "C") ///
    (connected coef event_time if scenario == "C"), ///
    yline(0) ///
    xline(-1) ///
    xtitle("Event Time") ///
    ytitle("Estimated Effect") ///
    title("Event-study Plot: Scenario C") ///
    legend(off)

graph export "figures/event_study_scenario_C.png", replace
graph drop _all

****************************************************
* 6. 场景 D：动态异质处理效应
****************************************************

quietly twoway ///
    (rcap ci_low ci_high event_time if scenario == "D") ///
    (connected coef event_time if scenario == "D"), ///
    yline(0) ///
    xline(-1) ///
    xtitle("Event Time") ///
    ytitle("Estimated Effect") ///
    title("Event-study Plot: Scenario D") ///
    legend(off)

graph export "figures/event_study_scenario_D.png", replace
graph drop _all





(file figures/event_study_scenario_A.png not found)
file figures/event_study_scenario_A.png saved as PNG format



(file figures/event_study_scenario_B.png not found)
file figures/event_study_scenario_B.png saved as PNG format



(file figures/event_study_scenario_C.png not found)
file figures/event_study_scenario_C.png saved as PNG format



(file figures/event_study_scenario_D.png not found)
file figures/event_study_scenario_D.png saved as PNG format



### 图像展示

下面展示导出的事件研究图：

#### Scenario A

![Scenario A](figures/event_study_scenario_A.png)

#### Scenario B

![Scenario B](figures/event_study_scenario_B.png)

#### Scenario C

![Scenario C](figures/event_study_scenario_C.png)

#### Scenario D

![Scenario D](figures/event_study_scenario_D.png)

## Part 5. Heterogeneity Analysis

本部分进行异质性分析。

根据作业要求，除了报告总体 ATT 外，还需要考察处理效应是否在不同处理批次或不同协变量子群之间存在差异。

本文进行两类异质性分析：

1. **Cohort heterogeneity**：分别估计第 4 期、第 6 期和第 8 期处理组的 ATT；
2. **Covariate subgroup heterogeneity**：按照二元协变量 $X_2$ 分组，分别估计 $X_2=0$ 和 $X_2=1$ 子样本中的 ATT。

其中，cohort heterogeneity 主要用于观察不同处理批次之间的效应差异。特别是在场景 D 中，DGP 明确设定了 cohort 异质性和动态衰减效应，因此不同 cohort 的真实 ATT 本来就不同。

协变量子群异质性则用于观察估计方法在不同可观测特征子样本中的表现。

### 5.1 Cohort Heterogeneity

本节分别估计不同处理批次的 ATT。

对于每一个场景和每一个 cohort，我们只保留：

- 当前 cohort 的处理组；
- never-treated 个体作为 clean control group。

然后估计如下 DID 模型：

$$
Y_{it} = \alpha_i + \lambda_t + \beta_{g} D_{it}^{g} + \varepsilon_{it}
$$

其中：

$$
D_{it}^{g}=1(G_i=g,\ t\geq g)
$$

估计得到的 $\beta_g$ 可以理解为当前 cohort 相对于 never-treated 控制组的 cohort-specific ATT。

In [22]:
****************************************************
* Part 5.1：Cohort 异质性分析
****************************************************

****************************************************
* 1. 读取示例 DGP 数据
****************************************************

use "example_dgp.dta", clear

quietly xtset id t

* 保存完整示例数据
tempfile base
save `base', replace

****************************************************
* 2. 创建 cohort 异质性结果矩阵
****************************************************

capture matrix drop cohort_heterogeneity_table

* 12 行 = 4 个场景 × 3 个 cohort
* 5 列分别为：真实 ATT、估计 ATT、标准误、偏误、处理组个体数
matrix cohort_heterogeneity_table = J(12, 5, .)

matrix colnames cohort_heterogeneity_table = ///
    True_ATT Estimate SE Bias N_Treated

matrix rownames cohort_heterogeneity_table = ///
    A_g4 A_g6 A_g8 ///
    B_g4 B_g6 B_g8 ///
    C_g4 C_g6 C_g8 ///
    D_g4 D_g6 D_g8

****************************************************
* 3. 循环估计不同场景和不同 cohort 的 ATT
****************************************************

local row = 1

foreach sc in A B C D {

    foreach cohort in 4 6 8 {

        ************************************************
        * 3.1 读取完整数据
        ************************************************

        use `base', clear

        ************************************************
        * 3.2 只保留当前 cohort 和 never-treated
        ************************************************

        keep if g_`sc' == `cohort' | g_`sc' == 0

        ************************************************
        * 3.3 生成当前 cohort 的处理状态
        ************************************************

        gen D_cohort = (g_`sc' == `cohort' & t >= `cohort')

        ************************************************
        * 3.4 计算当前 cohort 的真实 ATT
        ************************************************

        quietly summarize tau_`sc' if g_`sc' == `cohort' & t >= `cohort'
        local true_att = r(mean)

        ************************************************
        * 3.5 计算当前 cohort 的处理组个体数
        ************************************************

        quietly count if t == 1 & g_`sc' == `cohort'
        local n_treated = r(N)

        ************************************************
        * 3.6 估计 cohort-specific DID
        ************************************************

        quietly xtset id t

        quietly xtreg y_`sc' D_cohort i.t, fe vce(cluster id)

        local beta = _b[D_cohort]
        local se = _se[D_cohort]
        local bias = `beta' - `true_att'

        ************************************************
        * 3.7 保存结果
        ************************************************

        matrix cohort_heterogeneity_table[`row', 1] = `true_att'
        matrix cohort_heterogeneity_table[`row', 2] = `beta'
        matrix cohort_heterogeneity_table[`row', 3] = `se'
        matrix cohort_heterogeneity_table[`row', 4] = `bias'
        matrix cohort_heterogeneity_table[`row', 5] = `n_treated'

        local row = `row' + 1
    }
}

****************************************************
* 4. 输出 cohort 异质性结果
****************************************************

display "Cohort 异质性分析结果："
matrix list cohort_heterogeneity_table, format(%9.3f)





(file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000009.tmp not found)
file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_000009.tmp saved as .dta
    format






(2,500 observations deleted)
(2,500 observations deleted)
(2,500 observations deleted)
(2,500 observations deleted)
(2,500 observations deleted)
(2,500 observations deleted)
(2,500 observations deleted)
(2,500 observations deleted)
(2,500 observations deleted)
(2,500 observations deleted)
(2,500 observations deleted)
(2,500 observations deleted)

Cohort 异质性分析结果：


cohort_heterogeneity_table[12,5]
       True_ATT   Estimate         SE       Bias  N_Treated
A_g4      2.000      1.747      0.091     -0.253    125.000
A_g6      2.000      1.941      0.084     -0.059    125.000
A_g8      2.000      1.970      0.088     -0.030    125.000
B_g4      2.000      3.077      0.095      1.077    125.000
B_g6      2.000      2.715      0.087      0.715    125.000
B_g8      2.000      2.503      0.093      0.503    125.000
C_g4      2

### 5.2 Covariate Subgroup Heterogeneity

本节按照二元协变量 $X_2$ 进行子群异质性分析。

具体地，我们将样本分为：

- $X_2=0$；
- $X_2=1$。

然后在每一个子样本中，使用 Callaway and Sant'Anna DID 方法估计总体 ATT。

由于在子样本中 $X_2$ 是常数，因此估计时只控制连续协变量 $X_1$。

In [23]:
****************************************************
* Part 5.2：协变量子群异质性分析
****************************************************

****************************************************
* 1. 读取示例 DGP 数据
****************************************************

use "example_dgp.dta", clear

quietly xtset id t

* 保存完整示例数据
tempfile base
save `base', replace

****************************************************
* 2. 创建子群异质性结果矩阵
****************************************************

capture matrix drop subgroup_heterogeneity_table

* 8 行 = 4 个场景 × 2 个 x2 子群
* 5 列分别为：真实 ATT、CSDID 估计 ATT、标准误、偏误、已处理个体数
matrix subgroup_heterogeneity_table = J(8, 5, .)

matrix colnames subgroup_heterogeneity_table = ///
    True_ATT CSDID_ATT SE Bias N_Treated

matrix rownames subgroup_heterogeneity_table = ///
    A_x2_0 A_x2_1 ///
    B_x2_0 B_x2_1 ///
    C_x2_0 C_x2_1 ///
    D_x2_0 D_x2_1

****************************************************
* 3. 循环估计不同场景和不同 x2 子群的 ATT
****************************************************

local row = 1

foreach sc in A B C D {

    foreach group in 0 1 {

        ************************************************
        * 3.1 读取完整数据并保留当前子群
        ************************************************

        use `base', clear

        keep if x2 == `group'

        quietly xtset id t

        ************************************************
        * 3.2 计算当前子群中的真实 ATT
        ************************************************

        quietly summarize tau_`sc' if D_`sc' == 1
        local true_att = r(mean)

        ************************************************
        * 3.3 计算当前子群中的已处理个体数
        ************************************************

        quietly count if t == 1 & g_`sc' > 0
        local n_treated = r(N)

        ************************************************
        * 3.4 使用 CSDID 估计当前子群 ATT
        ************************************************

        * 由于 x2 在子样本中是常数，因此只控制 x1
        quietly csdid y_`sc' x1, ///
            ivar(id) ///
            time(t) ///
            gvar(g_`sc') ///
            method(dripw) ///
            notyet

        quietly estat simple

        matrix temp = r(table)

        local beta = temp[1,1]
        local se = temp[2,1]
        local bias = `beta' - `true_att'

        ************************************************
        * 3.5 保存结果
        ************************************************

        matrix subgroup_heterogeneity_table[`row', 1] = `true_att'
        matrix subgroup_heterogeneity_table[`row', 2] = `beta'
        matrix subgroup_heterogeneity_table[`row', 3] = `se'
        matrix subgroup_heterogeneity_table[`row', 4] = `bias'
        matrix subgroup_heterogeneity_table[`row', 5] = `n_treated'

        local row = `row' + 1
    }
}

****************************************************
* 4. 输出协变量子群异质性结果
****************************************************

display "协变量子群异质性分析结果："
matrix list subgroup_heterogeneity_table, format(%9.3f)





(file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_00000a.tmp not found)
file C:\Users\SMOOTH~1\AppData\Local\Temp\ST_8f88_00000a.tmp saved as .dta
    format






(2,690 observations deleted)
(2,310 observations deleted)
(2,690 observations deleted)
(2,310 observations deleted)
(2,690 observations deleted)
(2,310 observations deleted)
(2,690 observations deleted)
(2,310 observations deleted)

协变量子群异质性分析结果：


subgroup_heterogeneity_table[8,5]
         True_ATT  CSDID_ATT         SE       Bias  N_Treated
A_x2_0      2.000      1.911      0.118     -0.089    169.000
A_x2_1      2.000      1.646      0.116     -0.354    206.000
B_x2_0      2.000      1.496      0.175     -0.504    141.000
B_x2_1      2.000      2.394      0.141      0.394    234.000
C_x2_0      2.000      3.620      0.138      1.620    160.000
C_x2_1      2.000      3.771      0.160      1.771    215.000
D_x2_0      1.364      1.518      0.119      0.154    171.000
D_x2_1      1.371      1.317      0.101     -0.054  

### 5.3 异质性结果解释

cohort 异质性结果用于观察不同处理批次之间的 ATT 是否存在差异。

在场景 A、B、C 中，真实处理效应被设定为同质效应，即所有已处理观测的真实处理效应都等于 2。因此，如果不同 cohort 之间的估计结果存在差异，主要可能来自有限样本误差、处理组和控制组之间的可比性差异，或者估计方法本身的偏误。

在场景 D 中，DGP 明确设定了 cohort 异质性和动态衰减效应。第 4 期处理组、第 6 期处理组和第 8 期处理组的初始处理效应不同，并且处理效应会随着事件时间逐渐衰减。因此，场景 D 中不同 cohort 的真实 ATT 本身就应该不同。这个结果可以用来检验估计方法是否能够反映 DGP 中设定的 cohort 异质性。

协变量子群异质性结果用于比较 $X_2=0$ 和 $X_2=1$ 两个子样本中的 ATT 估计结果。

在场景 B 和场景 C 中，处理时间与协变量有关，因此按照 $X_2$ 分组有助于观察估计结果是否会因为可观测特征不同而发生变化。如果在真实处理效应同质的情况下，不同子群之间的估计值差异较大，说明估计结果可能受到样本选择、协变量不平衡或识别假设不满足的影响。

总体而言，异质性分析不仅可以展示不同 cohort 和不同协变量子群下的处理效应估计，也可以帮助判断估计方法是否能够准确反映 DGP 中设定的处理效应结构。

## Part 6. 统计性质验证

本部分根据 Monte Carlo 模拟结果，对作业要求中的统计性质命题进行验证。

前文已经完成了 DGP 设计、估计方法实现、Monte Carlo 主结果表、覆盖率表、偏误-方差权衡图和事件研究图。本部分在这些结果基础上，进一步围绕四个命题进行集中讨论：

1. 当存在 never-treated 单位时，Callaway and Sant'Anna 估计量是否一致；
2. 当平行趋势只在条件于协变量 $X_i$ 时成立，滚动匹配 DID 是否能够改善估计表现；
3. 合成控制法在什么条件下优于传统匹配；
4. 传统 TWFE 在交错处理下的偏误方向和偏误来源。

### 6.1 命题 1：CSDID 在存在 never-treated 单位时是否一致？

命题 1 关注的是：当数据中存在 never-treated 单位，并且识别假设成立时，Callaway and Sant'Anna 估计量是否能够一致估计真实 ATT。

在本文的 DGP 中：

- 场景 A 满足随机处理分配、平行趋势和同质处理效应；
- 场景 D 满足随机处理分配，但存在动态处理效应和 cohort 异质性；
- 场景 B 中处理分配依赖可观测协变量；
- 场景 C 中存在不可观测时变混淆。

因此，如果 CSDID 估计量表现良好，应该在场景 A 和场景 D 中表现出较小偏误；在场景 B 中，如果协变量调整充分，偏误也应小于传统 TWFE；而在场景 C 中，由于不可观测时变混淆破坏平行趋势，CSDID 也可能产生明显偏误。

In [1]:
****************************************************
* Part 6.1：命题 1 验证
****************************************************

****************************************************
* 1. 读取 Monte Carlo 主结果
****************************************************

use "mc_main_raw_results.dta", clear

****************************************************
* 2. 生成 CSDID 的误差指标
****************************************************

foreach sc in A B C D {

    capture drop err_`sc'_csdid
    capture drop abserr_`sc'_csdid
    capture drop sqerr_`sc'_csdid
    capture drop cover_`sc'_csdid

    * 估计误差
    gen err_`sc'_csdid = beta_`sc'_csdid - true_`sc'_csdid

    * 绝对误差
    gen abserr_`sc'_csdid = abs(err_`sc'_csdid)

    * 平方误差
    gen sqerr_`sc'_csdid = err_`sc'_csdid^2

    * 95% 置信区间覆盖率
    gen cover_`sc'_csdid = ///
        (true_`sc'_csdid >= beta_`sc'_csdid - 1.96 * se_`sc'_csdid ///
      &  true_`sc'_csdid <= beta_`sc'_csdid + 1.96 * se_`sc'_csdid)
}

****************************************************
* 3. 创建 CSDID 验证表
****************************************************

capture matrix drop csdid_validation_table

matrix csdid_validation_table = J(4, 7, .)

matrix colnames csdid_validation_table = ///
    True_ATT Mean_CSDID Bias RMSE MedAE Mean_SE Coverage

matrix rownames csdid_validation_table = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

****************************************************
* 4. 汇总四个场景下的 CSDID 表现
****************************************************

local row = 1

foreach sc in A B C D {

    quietly summarize true_`sc'_csdid
    local mean_true = r(mean)

    quietly summarize beta_`sc'_csdid
    local mean_beta = r(mean)

    quietly summarize err_`sc'_csdid
    local bias = r(mean)

    quietly summarize sqerr_`sc'_csdid
    local rmse = sqrt(r(mean))

    quietly summarize abserr_`sc'_csdid, detail
    local medae = r(p50)

    quietly summarize se_`sc'_csdid
    local mean_se = r(mean)

    quietly summarize cover_`sc'_csdid
    local coverage = r(mean)

    matrix csdid_validation_table[`row',1] = `mean_true'
    matrix csdid_validation_table[`row',2] = `mean_beta'
    matrix csdid_validation_table[`row',3] = `bias'
    matrix csdid_validation_table[`row',4] = `rmse'
    matrix csdid_validation_table[`row',5] = `medae'
    matrix csdid_validation_table[`row',6] = `mean_se'
    matrix csdid_validation_table[`row',7] = `coverage'

    local row = `row' + 1
}

****************************************************
* 5. 输出 CSDID 验证结果
****************************************************

display "命题 1：CSDID 统计性质验证结果"
matrix list csdid_validation_table, format(%9.3f)


(simulate: mc_main_once)

(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)
(1 missing value generated)







命题 1：CSDID 统计性质验证结果


csdid_validation_table[4,7]
              True_ATT  Mean_CSDID        Bias        RMSE       MedAE
Scenario_A       2.000       2.000      -0.000       0.083       0.054
Scenario_B       2.000       2.008       0.008       0.260       0.165
Scenario_C       2.000       3.473       1.473       1.477       1.475
Scenario_D       1.368       1.368      -0.000       0.080       0.055

               Mean_SE    Coverage
Scenario_A       0.081       0.937
Scenario_B       0.157       0.759
Scenario_C       0.109       0.000
Scenario_D       0.081       0.955


### 命题 1 结果解释

CSDID 的核心优势在于它针对不同处理批次和不同时期分别估计 $ATT(g,t)$，避免了传统 TWFE 在交错处理下可能出现的错误比较问题。

从 Monte Carlo 结果看，如果场景 A 和场景 D 中 CSDID 的 Bias 和 RMSE 较小，说明在存在 never-treated 单位且平行趋势成立时，CSDID 能够较好恢复真实 ATT。

如果场景 B 中 CSDID 的偏误小于 TWFE，说明在处理分配依赖可观测协变量时，协变量调整能够改善估计表现。

如果场景 C 中 CSDID 仍然出现明显偏误，则说明 CSDID 并不能自动解决不可观测时变混淆问题。也就是说，现代 DID 方法能够修正交错处理下的错误比较问题，但仍然依赖平行趋势等识别假设。

### 6.2 命题 2：条件平行趋势下滚动匹配 DID 是否优于无条件 DID？

命题 2 关注的是：当平行趋势只在条件于协变量 $X_i$ 时成立，滚动匹配 DID 是否能够改善估计表现。

在本文 DGP 中，场景 B 专门用于检验这一问题。场景 B 中：

- 处理分配与 $X_1$、$X_2$ 有关；
- $X_1$、$X_2$ 同时影响未处理结果趋势；
- 因此，无条件 DID 或 TWFE 可能因为协变量分布不平衡而产生偏误；
- 匹配类 DID 方法通过在协变量上进行匹配，有望减少这种偏误。

本节比较场景 B 中 TWFE、同期 PSM-DID、滚动 PSM-DID 和 CSDID 的 Monte Carlo 表现。

In [2]:
****************************************************
* Part 6.2：命题 2 验证
****************************************************

****************************************************
* 1. 读取 Monte Carlo 主结果
****************************************************

use "mc_main_raw_results.dta", clear

****************************************************
* 2. 生成场景 B 下各方法的误差指标
****************************************************

foreach m in twfe psm rpsm csdid {

    capture drop err_B_`m'
    capture drop abserr_B_`m'
    capture drop sqerr_B_`m'
    capture drop cover_B_`m'

    gen err_B_`m' = beta_B_`m' - true_B_`m'
    gen abserr_B_`m' = abs(err_B_`m')
    gen sqerr_B_`m' = err_B_`m'^2

    gen cover_B_`m' = ///
        (true_B_`m' >= beta_B_`m' - 1.96 * se_B_`m' ///
      &  true_B_`m' <= beta_B_`m' + 1.96 * se_B_`m')
}

****************************************************
* 3. 创建命题 2 验证表
****************************************************

capture matrix drop matching_validation_table

matrix matching_validation_table = J(4, 7, .)

matrix colnames matching_validation_table = ///
    True_ATT Mean_Estimate Bias RMSE MedAE Mean_SE Coverage

matrix rownames matching_validation_table = ///
    TWFE PSM_DID Rolling_PSM CSDID

****************************************************
* 4. 汇总场景 B 中各方法的表现
****************************************************

local row = 1

foreach m in twfe psm rpsm csdid {

    quietly summarize true_B_`m'
    local mean_true = r(mean)

    quietly summarize beta_B_`m'
    local mean_beta = r(mean)

    quietly summarize err_B_`m'
    local bias = r(mean)

    quietly summarize sqerr_B_`m'
    local rmse = sqrt(r(mean))

    quietly summarize abserr_B_`m', detail
    local medae = r(p50)

    quietly summarize se_B_`m'
    local mean_se = r(mean)

    quietly summarize cover_B_`m'
    local coverage = r(mean)

    matrix matching_validation_table[`row',1] = `mean_true'
    matrix matching_validation_table[`row',2] = `mean_beta'
    matrix matching_validation_table[`row',3] = `bias'
    matrix matching_validation_table[`row',4] = `rmse'
    matrix matching_validation_table[`row',5] = `medae'
    matrix matching_validation_table[`row',6] = `mean_se'
    matrix matching_validation_table[`row',7] = `coverage'

    local row = `row' + 1
}

****************************************************
* 5. 输出命题 2 验证结果
****************************************************

display "命题 2：条件平行趋势下匹配 DID 的表现"
matrix list matching_validation_table, format(%9.3f)


(simulate: mc_main_once)

(1 missing value generated)
(1 missing value generated)
(1 missing value generated)







命题 2：条件平行趋势下匹配 DID 的表现


matching_validation_table[4,7]
                 True_ATT  Mean_Estim~e          Bias          RMSE
       TWFE         2.000         2.392         0.392         0.396
    PSM_DID         2.000         2.004         0.004         0.227
Rolling_PSM         2.000         2.088         0.088         0.313
      CSDID         2.000         2.008         0.008         0.260

                    MedAE       Mean_SE      Coverage
       TWFE         0.391         0.057         0.000
    PSM_DID         0.156         0.215         0.913
Rolling_PSM         0.193         0.174         0.712
      CSDID         0.165         0.157         0.759


### 命题 2 结果解释

场景 B 是检验匹配方法优势的关键场景。由于处理分配依赖可观测协变量 $X_1$ 和 $X_2$，而这些协变量也影响结果趋势，因此传统 TWFE 可能出现系统性偏误。

如果滚动 PSM-DID 的 Bias、RMSE 和 Median Absolute Error 明显小于 TWFE，说明在平行趋势只在条件于协变量时成立的情况下，匹配方法能够改善估计表现。

同期 PSM-DID 只使用处理发生当期的信息，而滚动 PSM-DID 使用多个处理后时期，因此两者的目标参数略有不同。滚动 PSM-DID 更接近所有处理后时期上的平均 ATT。

如果场景 B 中 CSDID 也表现较好，说明协变量调整和 clean control 的构造同样有助于缓解由可观测选择导致的偏误。

### 6.3 命题 3：合成控制法在什么条件下优于传统匹配？

命题 3 关注的是：合成控制法 SCM 在什么条件下可能优于传统匹配方法。

传统匹配方法通常基于处理前协变量进行匹配，例如本文中的同期 PSM-DID 和滚动 PSM-DID 使用 $X_1$ 和 $X_2$ 进行倾向得分匹配。其核心思想是使处理组和控制组在可观测协变量上尽可能相似。

合成控制法的核心思想不同。SCM 不仅关注协变量相似性，更强调处理前结果路径的相似性。也就是说，SCM 希望通过 donor pool 中多个控制单位的加权组合，构造出一个在处理前走势上尽可能接近处理组的“合成控制组”。

因此，SCM 在以下条件下可能优于传统匹配方法：

1. 处理单位数量较少，甚至只有一个或少数几个处理单位；
2. donor pool 中存在足够多且足够相似的未处理单位；
3. 处理前时期较长，能够较好拟合处理前结果路径；
4. 处理前结果路径比静态协变量更能反映潜在结果趋势；
5. 处理组和控制组之间的差异可以通过 donor pool 的加权组合近似。

相反，如果处理前时期较短，或者 donor pool 中没有与处理组趋势相似的单位，SCM 的表现可能较差。

在本文 DGP 中，SCM 的表现取决于两个关键因素。

第一，处理前时期数量不同。对于 $G_i=4$ 的 cohort，处理前只有 3 期；对于 $G_i=6$ 的 cohort，处理前有 5 期；对于 $G_i=8$ 的 cohort，处理前有 7 期。因此，越晚接受处理的 cohort，SCM 可以利用的处理前信息越多，理论上越容易构造出较好的合成控制组。

第二，donor pool 的相似性也很重要。本文使用 never-treated 个体作为 donor pool。如果 never-treated 个体能够较好匹配处理组的处理前结果路径，SCM 估计会更可靠；如果处理组和 donor pool 在处理前趋势上差异较大，SCM 可能产生偏误。

因此，本文的 SCM 结果应当重点观察两个方面：

1. 不同 cohort 的 SCM 估计是否随着处理前时期增加而更稳定；
2. 场景 A 和 D 中，由于处理分配较为随机，SCM 应当比场景 C 更容易构造有效的合成控制组。

In [1]:
****************************************************
* Part 6.3：SCM 条件诊断
****************************************************

****************************************************
* 1. 读取示例 DGP 数据
****************************************************

use "example_dgp.dta", clear

****************************************************
* 2. 从数据中自动读取最大时期数
****************************************************

quietly summarize t
local T_max = r(max)

****************************************************
* 3. 创建 SCM 条件诊断表
****************************************************

capture matrix drop scm_condition_table

matrix scm_condition_table = J(3, 3, .)

matrix colnames scm_condition_table = Pre_Periods Post_Periods N_Treated

matrix rownames scm_condition_table = ///
    Cohort_g4 Cohort_g6 Cohort_g8

****************************************************
* 4. 计算不同 cohort 的处理前和处理后时期数量
****************************************************

local row = 1

foreach cohort in 4 6 8 {

    * 处理前时期数量：1 到 g-1
    local pre_periods = `cohort' - 1

    * 处理后时期数量：g 到 T
    local post_periods = `T_max' - `cohort' + 1

    * 这里以场景 A 为例，四个场景的 cohort 人数设计相同
    quietly count if t == 1 & g_A == `cohort'
    local n_treated = r(N)

    matrix scm_condition_table[`row',1] = `pre_periods'
    matrix scm_condition_table[`row',2] = `post_periods'
    matrix scm_condition_table[`row',3] = `n_treated'

    local row = `row' + 1
}

****************************************************
* 5. 输出 SCM 条件诊断表
****************************************************

display "SCM 适用条件诊断表："
matrix list scm_condition_table, format(%9.0f)











SCM 适用条件诊断表：


scm_condition_table[3,3]
            Pre_Periods  Post_Periods     N_Treated
Cohort_g4             3             7           125
Cohort_g6             5             5           125
Cohort_g8             7             3           125


### 命题 3 结果解释

从 SCM 条件诊断表可以看到，$G_i=4$ 的处理组只有 3 个处理前时期，而 $G_i=8$ 的处理组有 7 个处理前时期。处理前时期越长，SCM 越容易利用处理前结果路径构造有效的合成控制组。

因此，在本文设定中，SCM 对较晚处理的 cohort 理论上更有优势。对于较早处理的 cohort，由于处理前信息较少，合成控制组的拟合可能不够稳定。

与传统匹配方法相比，SCM 的优势在于它直接匹配处理前结果路径，而不仅仅匹配 $X_1$ 和 $X_2$。当处理前结果路径能够较好概括潜在结果趋势，并且 donor pool 中存在相似单位时，SCM 可能优于传统匹配。

但是，SCM 也有局限。它依赖 donor pool 的质量。如果 never-treated 个体与处理组在处理前趋势上差异较大，或者处理前时期太短，SCM 的估计结果可能反而不如简单的匹配 DID 稳定。

### 6.4 命题 4：传统 TWFE 在交错处理下的偏误方向和来源

命题 4 关注的是：传统 TWFE 在交错处理设定下为什么会产生偏误，以及偏误方向和大小由哪些因素决定。

传统 TWFE 模型为：

$$
Y_{it}=\alpha_i+\lambda_t+\beta D_{it}+\varepsilon_{it}
$$

在没有处理效应异质性、且平行趋势成立时，TWFE 可以较好估计平均处理效应。

但是在交错处理设定中，不同个体在不同时间接受处理。此时，TWFE 会隐含地使用多种 2×2 DID 比较，包括：

1. 早处理组 vs never-treated 组；
2. 晚处理组 vs never-treated 组；
3. 早处理组 vs 晚处理组；
4. 晚处理组 vs 早处理组。

其中，第 3 和第 4 类比较可能存在问题。特别是当已经接受处理的早处理组被用作晚处理组的对照组时，如果处理效应具有动态变化，就会产生错误比较。

根据 Goodman-Bacon 分解，TWFE 可以看作多个 2×2 DID 的加权平均。当处理效应存在动态异质性时，一些比较可能获得不合适的权重，甚至导致 TWFE 偏离真实 ATT。

In [2]:
****************************************************
* Part 6.4：TWFE 偏误方向和大小验证
****************************************************

****************************************************
* 1. 读取 Monte Carlo 原始结果
****************************************************

use "mc_main_raw_results.dta", clear

****************************************************
* 2. 生成 TWFE 误差指标
****************************************************

foreach sc in A B C D {

    capture drop err_`sc'_twfe
    capture drop abserr_`sc'_twfe
    capture drop sqerr_`sc'_twfe
    capture drop cover_`sc'_twfe

    * 估计误差
    gen err_`sc'_twfe = beta_`sc'_twfe - true_`sc'_twfe

    * 绝对误差
    gen abserr_`sc'_twfe = abs(err_`sc'_twfe)

    * 平方误差
    gen sqerr_`sc'_twfe = err_`sc'_twfe^2

    * 95% 置信区间覆盖率
    gen cover_`sc'_twfe = ///
        (true_`sc'_twfe >= beta_`sc'_twfe - 1.96 * se_`sc'_twfe ///
      &  true_`sc'_twfe <= beta_`sc'_twfe + 1.96 * se_`sc'_twfe)
}

****************************************************
* 3. 创建 TWFE 偏误验证表
****************************************************

capture matrix drop twfe_bias_validation_table

matrix twfe_bias_validation_table = J(4, 7, .)

matrix colnames twfe_bias_validation_table = ///
    True_ATT Mean_TWFE Bias RMSE MedAE Mean_SE Coverage

matrix rownames twfe_bias_validation_table = ///
    Scenario_A Scenario_B Scenario_C Scenario_D

****************************************************
* 4. 汇总四个场景下 TWFE 的表现
****************************************************

local row = 1

foreach sc in A B C D {

    quietly summarize true_`sc'_twfe
    local mean_true = r(mean)

    quietly summarize beta_`sc'_twfe
    local mean_beta = r(mean)

    quietly summarize err_`sc'_twfe
    local bias = r(mean)

    quietly summarize sqerr_`sc'_twfe
    local rmse = sqrt(r(mean))

    quietly summarize abserr_`sc'_twfe, detail
    local medae = r(p50)

    quietly summarize se_`sc'_twfe
    local mean_se = r(mean)

    quietly summarize cover_`sc'_twfe
    local coverage = r(mean)

    matrix twfe_bias_validation_table[`row',1] = `mean_true'
    matrix twfe_bias_validation_table[`row',2] = `mean_beta'
    matrix twfe_bias_validation_table[`row',3] = `bias'
    matrix twfe_bias_validation_table[`row',4] = `rmse'
    matrix twfe_bias_validation_table[`row',5] = `medae'
    matrix twfe_bias_validation_table[`row',6] = `mean_se'
    matrix twfe_bias_validation_table[`row',7] = `coverage'

    local row = `row' + 1
}

****************************************************
* 5. 输出 TWFE 偏误验证结果
****************************************************

display "命题 4：TWFE 偏误方向和大小验证结果"
matrix list twfe_bias_validation_table, format(%9.3f)


(simulate: mc_main_once)








命题 4：TWFE 偏误方向和大小验证结果


twfe_bias_validation_table[4,7]
             True_ATT  Mean_TWFE       Bias       RMSE      MedAE    Mean_SE
Scenario_A      2.000      1.999     -0.001      0.053      0.036      0.051
Scenario_B      2.000      2.392      0.392      0.396      0.391      0.057
Scenario_C      2.000      3.140      1.140      1.142      1.138      0.088
Scenario_D      1.368      1.713      0.345      0.349      0.345      0.058

             Coverage
Scenario_A      0.944
Scenario_B      0.000
Scenario_C      0.000
Scenario_D      0.000


### 命题 4 结果解释

TWFE 的偏误方向和大小取决于两个因素：

第一，是否存在平行趋势违背。  
在场景 B 中，处理分配与可观测协变量相关，而这些协变量也影响结果趋势。因此，TWFE 可能将协变量导致的趋势差异误认为处理效应。若处理组本身具有更高增长趋势，则 TWFE 会倾向于高估 ATT。

在场景 C 中，处理分配进一步受到不可观测因素 $\eta_i$ 的影响，而 $\eta_i$ 同时影响未处理结果趋势。因此，TWFE 的偏误通常会更严重，因为固定效应和时间固定效应无法控制这种不可观测的时变混淆。

第二，是否存在动态处理效应和 cohort 异质性。  
在场景 D 中，不同 cohort 的处理效应不同，并且处理效应会随事件时间衰减。传统 TWFE 会混合不同 cohort、不同时期的比较，并可能把已经接受处理的早处理组作为晚处理组的对照组。这类比较在动态处理效应存在时会产生错误的反事实。

因此，在本文 DGP 中，TWFE 的主要偏误来源可以概括为：

1. 场景 B：可观测协变量导致的趋势差异；
2. 场景 C：不可观测时变混淆；
3. 场景 D：动态处理效应、cohort 异质性和交错处理下的错误比较。

如果 Monte Carlo 结果显示场景 A 中 TWFE 偏误接近 0，而场景 B、C、D 中偏误明显增大，就说明本文 DGP 成功复现了传统 TWFE 在交错处理和非理想识别条件下的局限。

## Part 7. 理论推导：Callaway and Sant'Anna 的 ATT(g,t) 识别公式

本部分进行理论推导。

在交错处理设定中，不同个体可能在不同时间首次接受处理。因此，一个自然的处理效应对象不是单一的总体 ATT，而是特定处理批次在特定时期的组别-时间平均处理效应，即：

$$
ATT(g,t)
$$

本文选择推导 Callaway and Sant'Anna DID 中的 $ATT(g,t)$ 识别公式。该推导与前文使用的 `csdid` 方法直接对应，也有助于解释为什么现代 DID 方法在交错处理设定下通常比传统 TWFE 更合适。

### 7.1 基本定义

令 $G_i$ 表示个体 $i$ 的首次处理时间。

如果个体 $i$ 在第 $g$ 期首次接受处理，则：

$$
G_i = g
$$

如果个体从未接受处理，则：

$$
G_i = 0
$$

处理状态定义为：

$$
D_{it}=1(G_i>0,\ t\geq G_i)
$$

潜在结果定义为：

- $Y_{it}(1)$：个体 $i$ 在时期 $t$ 接受处理时的潜在结果；
- $Y_{it}(0)$：个体 $i$ 在时期 $t$ 未接受处理时的潜在结果。

观测结果为：

$$
Y_{it}=D_{it}Y_{it}(1)+(1-D_{it})Y_{it}(0)
$$

对于第 $g$ 期处理组，在时期 $t\geq g$ 的组别-时间平均处理效应定义为：

$$
ATT(g,t)=E[Y_{it}(1)-Y_{it}(0)\mid G_i=g]
$$

这个对象表示：第 $g$ 期开始接受处理的个体，在时期 $t$ 的平均处理效应。

### 7.2 识别假设

为了识别 $ATT(g,t)$，需要以下两个核心假设。

#### 假设 1：无预期效应

在真正接受处理之前，个体的结果不会受到未来处理的影响。

对于第 $g$ 期处理组，在 $t<g$ 时：

$$
Y_{it}=Y_{it}(0)
$$

也就是说，处理组在处理前的观测结果等于未处理潜在结果。

#### 假设 2：平行趋势假设

对于第 $g$ 期处理组和 never-treated 控制组，在没有处理的情况下，两组的未处理潜在结果变化趋势相同。

以 $g-1$ 作为第 $g$ 期处理组接受处理前的最后一期，则平行趋势假设可以写为：

$$
E[Y_{it}(0)-Y_{i,g-1}(0)\mid G_i=g]
=
E[Y_{it}(0)-Y_{i,g-1}(0)\mid G_i=0]
$$

该假设的含义是：如果第 $g$ 期处理组没有接受处理，那么它们从 $g-1$ 到 $t$ 的结果变化，应该与 never-treated 组在同一时期的结果变化相同。

### 7.3 识别公式推导

我们的目标是识别：

$$
ATT(g,t)=E[Y_{it}(1)-Y_{it}(0)\mid G_i=g]
$$

对于第 $g$ 期处理组，在时期 $t\geq g$，它们已经接受处理，因此观测结果满足：

$$
Y_{it}=Y_{it}(1)
$$

所以：

$$
ATT(g,t)
=
E[Y_{it}\mid G_i=g]
-
E[Y_{it}(0)\mid G_i=g]
$$

其中，$E[Y_{it}\mid G_i=g]$ 可以观测，但 $E[Y_{it}(0)\mid G_i=g]$ 是反事实结果，无法直接观测。

为了构造这个反事实结果，可以从处理前一期 $g-1$ 出发。因为在 $g-1$ 期，第 $g$ 期处理组尚未接受处理，根据无预期效应：

$$
Y_{i,g-1}=Y_{i,g-1}(0)
$$

因此：

$$
E[Y_{i,g-1}(0)\mid G_i=g]
=
E[Y_{i,g-1}\mid G_i=g]
$$

接下来，将第 $g$ 期处理组在时期 $t$ 的未处理潜在结果写成：

$$
E[Y_{it}(0)\mid G_i=g]
=
E[Y_{i,g-1}(0)\mid G_i=g]
+
E[Y_{it}(0)-Y_{i,g-1}(0)\mid G_i=g]
$$

根据平行趋势假设：

$$
E[Y_{it}(0)-Y_{i,g-1}(0)\mid G_i=g]
=
E[Y_{it}(0)-Y_{i,g-1}(0)\mid G_i=0]
$$

对于 never-treated 组，在所有时期都没有接受处理，因此：

$$
Y_{it}=Y_{it}(0)
$$

$$
Y_{i,g-1}=Y_{i,g-1}(0)
$$

所以：

$$
E[Y_{it}(0)-Y_{i,g-1}(0)\mid G_i=0]
=
E[Y_{it}-Y_{i,g-1}\mid G_i=0]
$$

将以上结果代入反事实表达式：

$$
E[Y_{it}(0)\mid G_i=g]
=
E[Y_{i,g-1}\mid G_i=g]
+
E[Y_{it}-Y_{i,g-1}\mid G_i=0]
$$

再代入 $ATT(g,t)$ 的定义：

$$
ATT(g,t)
=
E[Y_{it}\mid G_i=g]
-
\left[
E[Y_{i,g-1}\mid G_i=g]
+
E[Y_{it}-Y_{i,g-1}\mid G_i=0]
\right]
$$

整理可得：

$$
ATT(g,t)
=
E[Y_{it}-Y_{i,g-1}\mid G_i=g]
-
E[Y_{it}-Y_{i,g-1}\mid G_i=0]
$$

这就是基于 never-treated 控制组的组别-时间 DID 识别公式。

### 7.4 推导结果解释

最终得到的识别公式为：

$$
ATT(g,t)
=
E[Y_{it}-Y_{i,g-1}\mid G_i=g]
-
E[Y_{it}-Y_{i,g-1}\mid G_i=0]
$$

这个公式说明，$ATT(g,t)$ 可以通过一个局部 DID 比较识别：

1. 先计算第 $g$ 期处理组从处理前一期 $g-1$ 到时期 $t$ 的结果变化；
2. 再计算 never-treated 控制组在同一时间区间内的结果变化；
3. 两者之差就是第 $g$ 期处理组在时期 $t$ 的平均处理效应。

与传统 TWFE 不同，Callaway and Sant'Anna 方法不会把已经接受处理的个体错误地作为控制组，而是针对每一个处理批次和每一个时期构造 clean comparison。

因此，在存在交错处理和处理效应异质性时，CSDID 比传统 TWFE 更适合估计组别-时间处理效应。

### 7.5 与本文 Monte Carlo 结果的关系

上述理论推导可以解释本文 Monte Carlo 结果中的几个现象。

首先，在场景 A 中，处理分配是随机的，平行趋势成立，处理效应同质。因此，CSDID 能够较好识别真实 ATT，估计偏误较小，覆盖率也应接近理论水平。

其次，在场景 B 中，处理时间与可观测协变量相关。如果估计中能够适当控制或调整协变量，CSDID 的偏误应当小于传统 TWFE。

再次，在场景 C 中，存在不可观测时变混淆因素 $\eta_i$。由于 $\eta_i$ 同时影响处理分配和未处理结果趋势，平行趋势假设被破坏。因此，即使使用 CSDID，也无法完全消除偏误。

最后，在场景 D 中，处理效应具有动态衰减和 cohort 异质性。传统 TWFE 可能因为错误比较和异质处理效应产生偏误，而 CSDID 通过估计 $ATT(g,t)$，能够更自然地处理交错处理和动态异质效应。

因此，理论推导和 Monte Carlo 结果是一致的：现代 DID 方法能够解决交错处理下的错误比较问题，但仍然依赖平行趋势等核心识别假设。

## Part 8. 论文批判性评价

本部分选择对以下论文进行批判性评价：

Bailey, Martha J., Shuqiao Sun, and Brenden Timpe. 2021. “Prep School for Poor Kids: The Long-Run Impacts of Head Start on Human Capital and Economic Self-Sufficiency.” *American Economic Review*, 111(12): 3963–4001.

本文不进行完整数据复现，而是从交错处理 DID 和现代 DID 方法的角度，对该论文的识别策略、潜在局限和可补充检验进行方法层面的评价。

选择该论文的原因是：Head Start 项目在不同 county、不同时间逐步 rollout，天然具有 staggered treatment adoption 的特征。因此，该论文非常适合用于讨论传统 DID、交错处理、动态处理效应和 cohort 异质性等问题。

### 8.1 论文背景和主要发现

Bailey、Sun 和 Timpe（2021）研究 Head Start 项目的长期影响。Head Start 是美国面向低收入儿童的公共学前教育项目，该论文关注早期教育干预是否能够改善个体成年后的教育、人力资本和经济自立能力。

根据 American Economic Review 的文章摘要，该论文使用大规模限制性行政数据，并利用 1965 年至 1980 年间 Head Start 在不同 county 的逐步 rollout，以及入学年龄 cutoff 所带来的暴露差异，评估 Head Start 的长期影响。

论文发现，Head Start 显著提高了成年后的人力资本和经济自立能力。例如，文章摘要报告 Head Start 使受教育年限增加 0.65 年，高中完成率提高 2.7%，大学入学率提高 8.5%，大学完成率提高 39%。这些结果说明，面向低收入儿童的公共学前教育项目可能具有显著的长期回报。

### 8.2 识别策略概括

该论文的核心识别思路可以理解为：不同地区在不同时间获得 Head Start 项目，儿童是否在关键年龄阶段暴露于 Head Start 取决于其出生 cohort、所在 county 以及项目 rollout 时间。

因此，该研究具有典型的交错处理特征：

1. 不同 county 并非在同一时间引入 Head Start；
2. 不同出生 cohort 在儿童早期是否暴露于 Head Start 存在差异；
3. 处理强度和处理时点可能在 county 和 cohort 维度上变化；
4. 长期结果在成年后才被观测，因此处理效应可能具有明显的动态性和异质性。

从本文大作业的角度看，这类设计和本文模拟中的 staggered adoption setting 很相似。不同 county 的 Head Start rollout 可以类比为不同处理 cohort，而尚未 rollout 或尚未暴露的 county-cohort 可以作为潜在控制组。

### 8.3 论文的优势

该论文有几个明显优势。

第一，研究问题重要。Head Start 是美国重要的反贫困和儿童发展政策，评估其长期影响对于教育政策和社会福利政策都有现实意义。

第二，数据质量高。论文使用大规模限制性行政数据，能够追踪个体长期的人力资本和经济结果。相比短期考试成绩或小样本调查数据，行政数据能够更全面地衡量长期影响。

第三，识别设计具有政策含义。利用 Head Start 在不同 county 的 rollout 和儿童入学年龄 cutoff，可以将政策扩展过程转化为准实验变化，从而识别儿童早期项目暴露对长期结果的影响。

第四，结果关注长期影响。许多早期教育项目的短期影响可能会在几年后衰减，但该论文直接考察成年后的教育和经济自立能力，因此能够更好地评估公共学前教育投资的长期回报。

### 8.4 从交错处理 DID 角度看可能存在的问题

尽管该论文的研究设计非常有价值，但从现代 DID 和交错处理的角度看，仍然有几个值得进一步讨论的问题。

#### 8.4.1 处理效应可能存在动态异质性

Head Start 的影响可能不是一次性、固定不变的。不同年龄暴露、不同地区项目质量、不同家庭背景和不同出生 cohort 都可能导致处理效应不同。

如果处理效应随时间变化，或者不同 county-cohort 的处理效应存在异质性，那么传统 TWFE 估计可能将不同处理时点和不同处理效应混合在一起，导致估计量不再等于清晰定义的平均处理效应。

这与本文 DGP 中的场景 D 类似。在场景 D 中，处理效应被设定为动态衰减且存在 cohort 异质性。Monte Carlo 结果表明，在这种情况下，传统 TWFE 可能出现系统性偏误，而 CSDID 等现代 DID 方法更适合估计组别-时间层面的处理效应。

#### 8.4.2 控制组选择需要特别谨慎

在 staggered rollout 设计中，如果已经接受处理的地区或 cohort 被用作尚未处理组的对照组，就可能产生错误比较。

对于 Head Start 研究而言，较早 rollout 的 county 可能在后续时期已经受到项目影响。如果这些地区被用来作为较晚 rollout 地区的对照组，那么估计结果可能受到 already-treated controls 的影响。

因此，更稳妥的做法是明确使用 never-treated 或 not-yet-treated 单位作为 clean controls，并报告不同控制组定义下的稳健性检验。

#### 8.4.3 rollout 时间可能不是完全外生的

Head Start 项目的 rollout 可能与地区经济条件、地方政治能力、行政执行能力、贫困程度或教育资源有关。如果这些因素同时影响儿童长期教育和经济结果，那么简单的 DID 设计可能受到选择性 rollout 的影响。

这类似于本文 DGP 中的场景 B 和场景 C。场景 B 中，处理时点与可观测协变量有关；场景 C 中，处理时点与不可观测因素有关。Monte Carlo 结果显示，如果这些因素影响未处理结果趋势，传统 DID 可能产生偏误。

因此，该论文需要充分说明 rollout 时间与潜在结果趋势之间的关系，并通过协变量控制、匹配、事件研究图或 placebo test 来增强识别可信度。

#### 8.4.4 长期结果可能受到其他政策或地区冲击干扰

该论文关注成年后的长期结果，而从儿童早期到成年之间经历了很长时间。在这段时间内，个体所在地区可能经历其他教育政策、福利政策、经济结构变化或人口迁移。

如果这些长期地区冲击与 Head Start rollout 时间相关，那么估计的长期影响可能混合了 Head Start 本身和其他政策环境变化的影响。

因此，除了控制 county 固定效应和 cohort 固定效应外，还可以考虑加入地区线性趋势、地区特定 cohort 趋势，或者排除存在重大同期政策变化的地区进行稳健性检验。

### 8.5 可以补充的现代 DID 检验

结合本文大作业中的模拟结果，我认为该论文可以补充以下现代 DID 检验，以增强识别可信度。

#### 第一，补充 Callaway and Sant'Anna DID

可以按照 county rollout cohort 构造 $ATT(g,t)$，分别估计不同 rollout cohort 在不同时间的处理效应。这样可以避免传统 TWFE 将不同 cohort 和不同 event time 的效应混合在一起。

如果 CSDID 的估计结果与原文主结果方向一致，说明结论对现代 DID 估计方法稳健。

#### 第二，补充 Sun and Abraham event-study

可以使用 Sun and Abraham 方法绘制事件研究图，展示 Head Start rollout 前后的动态效应路径。

重点应观察：

1. rollout 前的 lead 系数是否接近 0；
2. rollout 后的处理效应是否逐步显现；
3. 不同 rollout cohort 的动态效应是否存在明显差异。

如果处理前 lead 系数明显偏离 0，则说明平行趋势假设可能存在问题。

#### 第三，补充匹配 DID 或加权 DID

由于 Head Start rollout 可能与 county 的贫困程度、人口结构、教育资源和地方治理能力有关，可以先按照处理前 county 特征进行匹配或加权，然后再进行 DID 估计。

这类似本文大作业中的滚动 PSM-DID。若匹配后估计结果更加接近现代 DID 结果，说明可观测选择偏误得到了部分缓解。

#### 第四，报告 cohort-specific ATT

该论文可以进一步报告不同 rollout cohort 的处理效应，而不是只报告总体平均效应。

如果不同 cohort 的效应差异很大，就说明总体 ATT 可能掩盖了重要的异质性。这一点对于政策评估尤其重要，因为 Head Start 在不同时期、不同地区的项目质量和实施强度可能不同。

### 8.6 与本文 Monte Carlo 结果的联系

本文的 Monte Carlo 结果可以帮助理解该论文可能面临的方法问题。

首先，场景 A 表明，当处理分配近似随机、平行趋势成立且处理效应同质时，TWFE 和现代 DID 方法都可以较好恢复真实 ATT。

其次，场景 B 表明，如果处理时点与可观测协变量相关，传统 TWFE 可能产生偏误，而匹配 DID 或控制协变量的现代 DID 方法能够改善估计表现。这对应 Head Start rollout 可能与 county 特征相关的情况。

再次，场景 C 表明，如果处理时点与不可观测时变因素相关，那么无论是 TWFE、匹配 DID 还是 CSDID，都可能无法完全解决偏误问题。这提醒我们，Head Start 的 rollout 是否受到不可观测地区趋势影响，是识别可信度的关键。

最后，场景 D 表明，在动态处理效应和 cohort 异质性存在时，传统 TWFE 可能出现系统性偏误。因此，对于 Head Start 这种长期政策影响研究，不能只依赖一个总体 TWFE 系数，而应当进一步报告 event-study 动态效应和 cohort-specific ATT。

### 8.7 小结

总体而言，Bailey、Sun 和 Timpe（2021）是一篇非常重要的 Head Start 长期效果评估论文。其优势在于研究问题重要、数据质量高、结果具有明确政策含义，并且利用了 Head Start rollout 和年龄 cutoff 所带来的准实验变化。

但是，从现代 DID 和交错处理的角度看，该类研究仍然需要特别关注处理效应异质性、动态效应、clean control 的选择以及 rollout 时间的外生性。

因此，在类似研究中，仅报告传统 TWFE 结果是不够的。更完整的实证设计应当结合 CSDID、Sun and Abraham event-study、匹配 DID、cohort-specific ATT 和 placebo test，从多个角度验证识别假设和结果稳健性。

本文前面的 Monte Carlo 模拟也说明：现代 DID 方法能够缓解交错处理下的错误比较问题，但不能自动解决不可观测时变混淆。因此，任何基于 staggered rollout 的政策评估，都需要同时重视估计方法和识别假设本身。